# F1 Lap Time Prediction: Classical vs Quantum ML

**Project**: Predicting Formula 1 Lap Times Using Classical and Quantum Machine Learning: A Comparative Study Using FastF1 Telemetry Data

This notebook is optimized for Google Colab A100 High RAM.

## Step 1: Setup Runtime

1. Go to **Runtime** → **Change runtime type**
2. Set **Hardware accelerator**: A100 GPU
3. Set **Runtime shape**: High-RAM
4. Click **Save**

## Step 2: Install Dependencies

In [1]:
# Install dependencies
import sys
import subprocess
import importlib
import os

def install_package(package):
    """Try to install a package."""
    package_name = package.split('>=')[0].split('==')[0].strip()
    try:
        __import__(package_name)
        return True
    except ImportError:
        try:
            # Try using subprocess first
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package],
                                stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            # Try to import after installation
            importlib.import_module(package_name)
            return True
        except Exception:
            # If subprocess fails, try IPython magic (for Jupyter/Colab)
            try:
                if 'get_ipython' in globals() or 'get_ipython' in dir(__builtins__):
                    get_ipython().system(f'pip install -q {package}')
                    importlib.import_module(package_name)
                    return True
            except:
                pass
            return False

def check_and_install_requirements():
    """Install from requirements.txt if it exists."""
    if os.path.exists("requirements.txt"):
        try:
            print("Found requirements.txt, installing...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                                     stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
                print("Installed from requirements.txt")
                return True
            except:
                try:
                    if 'get_ipython' in globals() or 'get_ipython' in dir(__builtins__):
                        get_ipython().system('pip install -q -r requirements.txt')
                        print("Installed from requirements.txt")
                        return True
                except:
                    pass
        except Exception as e:
            print(f"Could not install from requirements.txt: {e}")
    return False

if not check_and_install_requirements():
    print("Installing dependencies...")
    packages = [
        "fastf1>=3.0.0",
        "pandas>=2.0.0",
        "numpy>=1.24.0",
        "scikit-learn>=1.3.0",
        "matplotlib>=3.7.0",
        "seaborn>=0.12.0",
        "qiskit>=0.45.0",
        "qiskit-machine-learning>=0.7.0",
        "qiskit-algorithms>=0.2.0",
        "qiskit-aer>=0.13.0",
        "psutil>=5.9.0",
        "ydata-profiling>=4.0.0",
        "tqdm",
        "scipy>=1.10.0"
    ]

    failed = []
    for pkg in packages:
        if not install_package(pkg):
            failed.append(pkg)

    if failed:
        print(f"\nWarning: Some packages failed: {failed}")
        print("Install manually if needed:")
        for pkg in failed:
            print(f"  !pip install {pkg}")
    else:
        print("All dependencies installed")

# Quick check of imports
print("\nChecking imports...")
try:
    import fastf1, pandas, numpy, sklearn, qiskit, matplotlib, seaborn
    print("All critical packages available")
except ImportError as e:
    print(f"Missing package: {e}")
    print("Install manually if needed")

print("\n" + "="*60)
print("Setup complete")
print("="*60)

Installing dependencies...

Install manually if needed:
  !pip install scikit-learn>=1.3.0
  !pip install qiskit-machine-learning>=0.7.0
  !pip install qiskit-algorithms>=0.2.0
  !pip install qiskit-aer>=0.13.0
  !pip install ydata-profiling>=4.0.0

Checking imports...
All critical packages available

Setup complete


## Step 3: Import All Code

In [2]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

import fastf1
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from typing import List, Optional, Tuple, Dict, Any
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkRegressor
from tqdm.auto import tqdm
import sys
import os
import pickle
import time
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

try:
    from qiskit.primitives import Estimator
except ImportError:
    try:
        from qiskit_aer.primitives import Estimator
    except ImportError:
        Estimator = None

print("Libraries imported")

# Capture exact versions for Appendix D2 reproducibility evidence
import qiskit
import qiskit_machine_learning
print(f"fastf1 version: {fastf1.__version__}")
print(f"qiskit version: {qiskit.__version__}")
print(f"qiskit-machine-learning version: {qiskit_machine_learning.__version__}")

Libraries imported
fastf1 version: 3.8.1
qiskit version: 2.3.1
qiskit-machine-learning version: 0.9.0


### Data Pipeline Functions

In [3]:
def _timedelta_to_seconds(series):
    """Convert timedelta to seconds."""
    return series.dt.total_seconds().astype(float)

def _normalize_time_features(df):
    """Create sector ratios and differences."""
    df = df.copy()
    if all(col in df.columns for col in ['Sector1Time_sec', 'Sector2Time_sec', 'Sector3Time_sec', 'LapTime_sec']):
        valid_mask = df['LapTime_sec'] > 0
        df.loc[valid_mask, 'Sector1Ratio'] = df.loc[valid_mask, 'Sector1Time_sec'] / df.loc[valid_mask, 'LapTime_sec']
        df.loc[valid_mask, 'Sector2Ratio'] = df.loc[valid_mask, 'Sector2Time_sec'] / df.loc[valid_mask, 'LapTime_sec']
        df.loc[valid_mask, 'Sector3Ratio'] = df.loc[valid_mask, 'Sector3Time_sec'] / df.loc[valid_mask, 'LapTime_sec']
    if all(col in df.columns for col in ['Sector1Time_sec', 'Sector2Time_sec', 'Sector3Time_sec']):
        df['Sector1vs2'] = df['Sector1Time_sec'] - df['Sector2Time_sec']
        df['Sector2vs3'] = df['Sector2Time_sec'] - df['Sector3Time_sec']
        df['Sector1vs3'] = df['Sector1Time_sec'] - df['Sector3Time_sec']
    return df

def _create_derived_features(df):
    """Add speed aggregations, position change, stint length."""
    df = df.copy()
    speed_cols = ['SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST']
    existing_speed_cols = [col for col in speed_cols if col in df.columns]
    if len(existing_speed_cols) > 1:
        df['AvgSpeed'] = df[existing_speed_cols].mean(axis=1)
        df['MaxSpeed'] = df[existing_speed_cols].max(axis=1)
        df['MinSpeed'] = df[existing_speed_cols].min(axis=1)
        df['SpeedRange'] = df['MaxSpeed'] - df['MinSpeed']
    if 'Position' in df.columns and 'Driver' in df.columns:
        df = df.sort_values(['Driver', 'LapNumber'])
        df['PositionChange'] = df.groupby('Driver')['Position'].diff()
    if 'Stint' in df.columns and 'Driver' in df.columns:
        df['StintLength'] = df.groupby(['Driver', 'Stint']).cumcount() + 1
    return df

def _handle_missing_values(df, strategy='median', threshold=0.5):
    """Drop high-missing columns, impute the rest."""
    df = df.copy()
    missing_ratio = df.isnull().sum() / len(df)
    cols_to_drop = missing_ratio[missing_ratio > threshold].index
    if len(cols_to_drop) > 0:
        df = df.drop(columns=cols_to_drop)
    if strategy == 'median':
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
    elif strategy == 'forward_fill':
        df = df.ffill().bfill()
    return df

def clean_single_session(session, drop_outliers=True, min_laps_per_driver=5):
    """Clean F1 session data - remove invalid laps, outliers, create features."""
    if not hasattr(session, 'laps') or session.laps is None or len(session.laps) == 0:
        return pd.DataFrame(), pd.Series(dtype=float), pd.DataFrame()

    laps = session.laps.copy()
    initial_count = len(laps)

    # Filter invalid laps
    deleted_mask = laps['Deleted'].fillna(False).astype(bool)
    accurate_mask = laps['IsAccurate'].fillna(True).astype(bool)
    generated_mask = laps['FastF1Generated'].fillna(False).astype(bool)

    laps = laps[
        laps['LapTime'].notna() &
        (~deleted_mask) &
        (accurate_mask) &
        (~generated_mask)
    ]

    laps = laps[laps['PitInTime'].isna() & laps['PitOutTime'].isna()]

    if 'Driver' in laps.columns:
        counts = laps['Driver'].value_counts()
        valid_drivers = counts[counts >= min_laps_per_driver].index.tolist()
        laps = laps[laps['Driver'].isin(valid_drivers)]

    if len(laps) == 0:
        return pd.DataFrame(), pd.Series(dtype=float), pd.DataFrame()

    time_cols = [
        'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
        'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
        'Time', 'LapStartTime'
    ]
    for col in time_cols:
        if col in laps.columns and pd.api.types.is_timedelta64_dtype(laps[col]):
            laps[col + '_sec'] = _timedelta_to_seconds(laps[col])

    numeric_cols_to_cast = [
        'LapNumber', 'Stint', 'TyreLife', 'Position',
        'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST'
    ]
    for col in numeric_cols_to_cast:
        if col in laps.columns:
            laps[col] = pd.to_numeric(laps[col], errors='coerce')

    critical_cols = ['LapTime_sec', 'Sector1Time_sec', 'Sector2Time_sec', 'Sector3Time_sec']
    critical_cols = [c for c in critical_cols if c in laps.columns]
    laps = laps.dropna(subset=critical_cols)

    if drop_outliers and 'LapTime_sec' in laps.columns:
        def remove_outliers(group):
            lt = group['LapTime_sec']
            med = lt.median()
            iqr = lt.quantile(0.75) - lt.quantile(0.25)
            if iqr == 0:
                return group[(lt >= med - 5.0) & (lt <= med + 5.0)]
            lower, upper = med - 1.5 * iqr, med + 1.5 * iqr
            return group[(lt >= lower) & (lt <= upper)]
        # Try with include_groups (pandas 2.0+), fallback to older syntax if it fails
        try:
            laps = laps.groupby('Driver', group_keys=False, include_groups=False).apply(remove_outliers)
        except TypeError:
            # Fallback for older pandas versions that don't support include_groups
            laps = laps.groupby('Driver', group_keys=False).apply(remove_outliers)

    laps = _create_derived_features(laps)
    laps = _normalize_time_features(laps)
    laps = _handle_missing_values(laps, strategy='median', threshold=0.5)

    categorical_cols = ['Driver', 'Team', 'Compound', 'TrackStatus']
    categorical_cols = [c for c in categorical_cols if c in laps.columns]
    laps_categorical = laps[categorical_cols].astype('category')
    dummies = pd.get_dummies(laps_categorical, prefix=categorical_cols, dummy_na=False)

    numeric_features = [
        'LapNumber', 'Stint', 'TyreLife',
        'Sector1Time_sec', 'Sector2Time_sec', 'Sector3Time_sec',
        'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'Position'
    ]
    derived_features = [
        'Sector1Ratio', 'Sector2Ratio', 'Sector3Ratio',
        'Sector1vs2', 'Sector2vs3', 'Sector1vs3',
        'AvgSpeed', 'MaxSpeed', 'MinSpeed', 'SpeedRange',
        'PositionChange', 'StintLength'
    ]
    for feat in derived_features:
        if feat in laps.columns:
            numeric_features.append(feat)
    numeric_features = [c for c in numeric_features if c in laps.columns]
    X_numeric = laps[numeric_features].copy()
    X_numeric = X_numeric.fillna(0.0)

    if len(dummies) > 0 and len(X_numeric) > 0:
        X_raw = pd.concat([X_numeric, dummies], axis=1)
    elif len(X_numeric) > 0:
        X_raw = X_numeric
    elif len(dummies) > 0:
        X_raw = dummies
    else:
        return pd.DataFrame(), pd.Series(dtype=float), pd.DataFrame()

    X_raw = X_raw.fillna(0.0)

    if X_raw.isnull().any().any():
        print(f"  Warning: Still have NaNs in X_raw. Dropping columns with all NaNs...")
        X_raw = X_raw.dropna(axis=1, how='all')
        X_raw = X_raw.fillna(0.0)
    y = laps['LapTime_sec'].astype(float)
    meta_cols = ['Driver', 'Team', 'LapNumber', 'Compound', 'TyreLife', 'Position']
    meta_cols = [c for c in meta_cols if c in laps.columns]
    meta = laps[meta_cols].reset_index(drop=True)
    return X_raw.reset_index(drop=True), y.reset_index(drop=True), meta

def load_sessions_and_build_dataset(years: List[int], session_name: str = "R",
                                     cache_dir: str = "cache", max_events_per_year: Optional[int] = None,
                                     quantum_n_components: int = 4, use_cache: bool = True,
                                     min_laps_per_driver: int = 5):
    """Load multiple F1 sessions and build feature matrices for classical and quantum models."""
    cache_file = Path(cache_dir) / f"preprocessed_{min(years)}_{max(years)}_{session_name}_{quantum_n_components}.pkl"
    if use_cache and cache_file.exists():
        print(f"Loading cached preprocessed data from {cache_file}...")
        try:
            with open(cache_file, 'rb') as f:
                cached_data = pickle.load(f)
                print("Cached data loaded successfully")
                return cached_data
        except Exception as e:
            print(f"Warning: Could not load cache ({e}), reprocessing...")

    Path(cache_dir).mkdir(parents=True, exist_ok=True)
    fastf1.Cache.enable_cache(cache_dir)
    all_X, all_y, all_meta, all_groups = [], [], [], []

    # Track statistics for diagnostics
    stats = {
        'years_processed': 0,
        'years_failed': 0,
        'sessions_attempted': 0,
        'sessions_successful': 0,
        'sessions_failed': 0,
        'sessions_no_data': 0,
        'rate_limit_encounters': 0,
        'failed_sessions': []
    }

    try:
        from fastf1.req import RateLimitExceededError
    except ImportError:
        RateLimitExceededError = None

    try:
        fastf1_version = tuple(map(int, fastf1.__version__.split('.')))
    except (AttributeError, ValueError):
        fastf1_version = (3, 0, 0)

    def get_schedule_with_retry(year, max_retries=3, base_delay=2):
        """Get event schedule with retry logic for rate limits."""
        for attempt in range(max_retries):
            try:
                schedule = fastf1.get_event_schedule(year)
                return schedule
            except Exception as e:
                error_str = str(e)
                is_rate_limit = (RateLimitExceededError is not None and isinstance(e, RateLimitExceededError)) or \
                               "RateLimit" in error_str or "500 calls" in error_str or "rate limit" in error_str.lower()

                if is_rate_limit and attempt < max_retries - 1:
                    stats['rate_limit_encounters'] += 1
                    delay = base_delay * (2 ** attempt)
                    print(f"  Rate limit hit for {year}, waiting {delay}s before retry {attempt + 1}/{max_retries}...")
                    time.sleep(delay)
                    continue
                else:
                    raise

    for year in tqdm(years, desc="Loading years"):
        try:
            schedule = get_schedule_with_retry(year)
            time.sleep(0.5)
            stats['years_processed'] += 1
        except (ValueError, KeyError, AttributeError) as e:
            print(f"  Warning: Could not get schedule for {year}: {e}")
            stats['years_failed'] += 1
            continue
        except Exception as e:
            error_str = str(e)
            is_rate_limit = (RateLimitExceededError is not None and isinstance(e, RateLimitExceededError)) or \
                           "RateLimit" in error_str or "500 calls" in error_str or "rate limit" in error_str.lower()

            if is_rate_limit:
                stats['rate_limit_encounters'] += 1
                print(f"Rate limit exceeded for {year}. Waiting 60 seconds...")
                print(f"     FastF1 API limit: 500 calls/hour. Consider using cached data or reducing years.")
                time.sleep(60)
                try:
                    schedule = fastf1.get_event_schedule(year)
                    time.sleep(0.5)
                    stats['years_processed'] += 1
                except Exception as e2:
                    print(f"  Error: Still cannot get schedule for {year} after wait: {e2}")
                    print(f"  Skipping {year} and continuing with other years...")
                    stats['years_failed'] += 1
                    continue
            else:
                print(f"  Error getting schedule for {year}: {e}")
                stats['years_failed'] += 1
                continue

        # Use per-year max events if max_events_per_year is None or if we need per-year limits
        year_max_events = get_max_events_per_year(year) if max_events_per_year is None else max_events_per_year
        if year_max_events is not None:
            schedule = schedule.iloc[:year_max_events]
        for _, ev in tqdm(schedule.iterrows(), total=len(schedule), desc=f"  {year} events", leave=False):
            gp_name = ev['EventName']
            stats['sessions_attempted'] += 1
            try:
                sess = fastf1.get_session(year, gp_name, session_name)

                try:
                    if fastf1_version >= (3, 0, 0):
                        try:
                            sess.load(laps=True, telemetry=False, weather=False, messages=False)
                        except TypeError:
                            sess.load(laps=True, telemetry=False, weather=False)
                    elif fastf1_version >= (2, 3, 0):
                        sess.load(laps=True, telemetry=False, weather=False)
                    else:
                        sess.load(with_telemetry=False, weather=False)
                except Exception as load_error:
                    error_str = str(load_error)
                    is_rate_limit = (RateLimitExceededError is not None and isinstance(load_error, RateLimitExceededError)) or \
                                   "RateLimit" in error_str or "500 calls" in error_str or "rate limit" in error_str.lower()

                    if is_rate_limit:
                        stats['rate_limit_encounters'] += 1
                        print(f"Rate limit during session load. Waiting 60 seconds...")
                        time.sleep(60)
                        try:
                            if fastf1_version >= (3, 0, 0):
                                try:
                                    sess.load(laps=True, telemetry=False, weather=False, messages=False)
                                except TypeError:
                                    sess.load(laps=True, telemetry=False, weather=False)
                            elif fastf1_version >= (2, 3, 0):
                                sess.load(laps=True, telemetry=False, weather=False)
                            else:
                                sess.load(with_telemetry=False, weather=False)
                        except Exception as e2:
                            print(f"  Error: Still cannot load session after wait: {e2}")
                            stats['sessions_failed'] += 1
                            stats['failed_sessions'].append(f"{year} {gp_name} {session_name}: {str(e2)[:100]}")
                            raise
                    else:
                        raise

                time.sleep(0.3)

                X_raw, y, meta = clean_single_session(sess, min_laps_per_driver=min_laps_per_driver)
                if len(X_raw) == 0:
                    stats['sessions_no_data'] += 1
                    continue

                stats['sessions_successful'] += 1
                group_id = f"{year}_{gp_name}_{session_name}"
                all_groups.extend([group_id] * len(X_raw))
                all_X.append(X_raw)
                all_y.append(y)
                meta = meta.copy()
                meta['Year'], meta['EventName'], meta['Session'] = year, gp_name, session_name
                all_meta.append(meta)
            except (ValueError, KeyError, AttributeError) as e:
                stats['sessions_failed'] += 1
                stats['failed_sessions'].append(f"{year} {gp_name} {session_name}: {str(e)[:100]}")
                continue
            except (ConnectionError, TimeoutError) as e:
                stats['sessions_failed'] += 1
                stats['failed_sessions'].append(f"{year} {gp_name} {session_name}: {str(e)[:100]}")
                continue
            except Exception as e:
                stats['sessions_failed'] += 1
                stats['failed_sessions'].append(f"{year} {gp_name} {session_name}: {str(e)[:100]}")
                continue

    if not all_X:
        error_msg = "No data collected. Diagnostics:\n"
        error_msg += f"  - Years processed: {stats['years_processed']}/{len(years)}\n"
        error_msg += f"  - Years failed: {stats['years_failed']}\n"
        error_msg += f"  - Sessions attempted: {stats['sessions_attempted']}\n"
        error_msg += f"  - Sessions successful: {stats['sessions_successful']}\n"
        error_msg += f"  - Sessions failed: {stats['sessions_failed']}\n"
        error_msg += f"  - Sessions with no valid data: {stats['sessions_no_data']}\n"
        error_msg += f"  - Rate limit encounters: {stats['rate_limit_encounters']}\n"
        if stats['failed_sessions']:
            error_msg += f"\n  Failed sessions (first 5):\n"
            for failed in stats['failed_sessions'][:5]:
                error_msg += f"    - {failed}\n"
        error_msg += "\nPossible solutions:\n"
        error_msg += "  - Rate limit exceeded (FastF1 API: 500 calls/hour) - wait 1 hour and retry\n"
        error_msg += "  - Check years/session_name are valid (e.g., session_name='R' for race)\n"
        error_msg += "  - Try using cached data: set use_cache=True\n"
        error_msg += "  - Reduce number of years or max_events_per_year\n"
        error_msg += "  - Check internet connection and FastF1 API status\n"
        raise RuntimeError(error_msg)

    X_raw_full = pd.concat(all_X, axis=0, sort=False).reset_index(drop=True)
    y_full = pd.concat(all_y, axis=0).reset_index(drop=True)
    meta_full = pd.concat(all_meta, axis=0).reset_index(drop=True)
    groups = pd.Series(all_groups, name="session_id")

    print(f"\nDataset: {len(X_raw_full)} samples, {len(X_raw_full.columns)} features")
    print(f"Successfully loaded {stats['sessions_successful']} sessions from {stats['years_processed']} years")

    # Handle any remaining NaN values
    if X_raw_full.isnull().any().any():
        X_raw_full = X_raw_full.fillna(0.0)
        # If still have NaNs after fillna, drop those rows
        if X_raw_full.isnull().any().any():
            valid_idx = X_raw_full.dropna().index
            X_raw_full = X_raw_full.iloc[valid_idx].reset_index(drop=True)
            y_full = y_full.iloc[valid_idx].reset_index(drop=True)
            meta_full = meta_full.iloc[valid_idx].reset_index(drop=True)
            groups = groups.iloc[valid_idx].reset_index(drop=True)

    if len(X_raw_full) == 0:
        raise RuntimeError("No valid data after NaN removal. Check data quality.")

    if X_raw_full.isnull().any().any():
        raise RuntimeError("Data still contains NaNs after cleaning. Cannot proceed.")

    # Scale features for classical models
    standard_scaler = StandardScaler()
    X_classical = standard_scaler.fit_transform(X_raw_full)
    X_classical = np.nan_to_num(X_classical, nan=0.0)  # Handle any NaN values
    X_classical = pd.DataFrame(X_classical, columns=X_raw_full.columns)

    # Reduce dimensionality and scale for quantum models
    pca = PCA(n_components=quantum_n_components)
    X_pca = pca.fit_transform(X_classical)
    X_pca = np.nan_to_num(X_pca, nan=0.0)  # Handle any NaN values

    quantum_scaler = MinMaxScaler(feature_range=(0.0, np.pi))
    X_quantum = quantum_scaler.fit_transform(X_pca)
    X_quantum = np.nan_to_num(X_quantum, nan=0.0)  # Handle any NaN values

    scalers = {"standard": standard_scaler, "pca": pca, "quantum": quantum_scaler}
    result = (X_classical, X_quantum, y_full, meta_full, scalers, groups)

    if use_cache:
        try:
            cache_file.parent.mkdir(parents=True, exist_ok=True)
            with open(cache_file, 'wb') as f:
                pickle.dump(result, f)
            print(f"Preprocessed data cached to {cache_file}")
        except Exception as e:
            print(f"Warning: Could not save cache ({e})")

    return result

print("Data pipeline functions loaded")

Data pipeline functions loaded


### Classical ML Functions

In [4]:
def evaluate_classical_models(years=None, session_name="R", cache_dir="cache",
                              max_events_per_year=None, quantum_n_components=4, n_splits=5,
                              save_plots=True, hgb_max_iter=300, min_laps_per_driver=5,
                              max_samples=None):
    """Evaluate classical ML models using group-based cross-validation."""
    if years is None:
        from datetime import datetime
        years = list(range(2020, min(2025, datetime.now().year + 1)))

    X_classical, _, y, meta, _, groups = load_sessions_and_build_dataset(
        years=years, session_name=session_name, cache_dir=cache_dir,
        max_events_per_year=max_events_per_year, quantum_n_components=quantum_n_components,
        min_laps_per_driver=min_laps_per_driver)

    # Apply same sample size limit as quantum models for fair comparison
    if max_samples is not None and len(X_classical) > max_samples:
        rng = np.random.RandomState(42)  # Same seed as quantum sampling
        idx = rng.choice(len(X_classical), size=max_samples, replace=False)
        idx = np.sort(idx)
        X_classical = X_classical.iloc[idx].reset_index(drop=True)
        y = y.iloc[idx].reset_index(drop=True)
        meta = meta.iloc[idx].reset_index(drop=True)
        groups = groups.iloc[idx].reset_index(drop=True)
        print(f"Applied max_samples={max_samples} limit for fair comparison with quantum models")

    print(f"Evaluating classical models: {len(X_classical)} samples, {n_splits} CV folds")

    # Standard classical models with full feature space
    models = {
        "ridge": Ridge(alpha=1.0),
        "hgb": HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=hgb_max_iter,
                                            l2_regularization=0.0, loss="squared_error", random_state=42,
                                            verbose=0)
    }

    # Add PCA-reduced baseline for fairer comparison with quantum models
    # This uses the same dimensionality reduction as quantum models
    if quantum_n_components is not None and quantum_n_components < len(X_classical.columns):
        from sklearn.decomposition import PCA
        pca = PCA(n_components=quantum_n_components, random_state=42)
        X_classical_pca = pca.fit_transform(X_classical)
        X_classical_pca = pd.DataFrame(X_classical_pca,
                                       columns=[f"PC{i+1}" for i in range(quantum_n_components)])
        models["ridge_pca"] = Ridge(alpha=1.0)  # Ridge with PCA-reduced features
        print(f"  Added PCA-reduced baseline (n_components={quantum_n_components}) for fair comparison")
    else:
        X_classical_pca = None

    gkf = GroupKFold(n_splits=n_splits)
    all_results, all_errors = [], []

    for name, model_template in models.items():
        print(f"Training {name.upper()}...")
        # Use PCA-reduced features for PCA models, full features for others
        X_data = X_classical_pca if name.endswith("_pca") and X_classical_pca is not None else X_classical
        splits = list(gkf.split(X_data, y, groups))
        for fold_id, (train_idx, test_idx) in enumerate(tqdm(splits, desc=f"  {name} folds", leave=False), 1):
            model = clone(model_template)
            X_tr, X_te = X_data.iloc[train_idx], X_data.iloc[test_idx]
            y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
            meta_te = meta.iloc[test_idx].reset_index(drop=True)
            groups_te = groups.iloc[test_idx].reset_index(drop=True)

            t0 = time.perf_counter()
            model.fit(X_tr, y_tr)
            t_train = time.perf_counter() - t0
            t0 = time.perf_counter()
            y_pred = model.predict(X_te)
            t_infer = time.perf_counter() - t0

            mae = mean_absolute_error(y_te, y_pred)
            rmse = np.sqrt(mean_squared_error(y_te, y_pred))  # Calculate RMSE manually for compatibility
            r2 = r2_score(y_te, y_pred)

            all_results.append({"model": name, "fold": fold_id, "MAE": mae, "RMSE": rmse, "R2": r2,
                               "train_time_s": t_train, "infer_time_s": t_infer, "n_train": len(X_tr), "n_test": len(X_te)})
            err_df = pd.DataFrame({"model": name, "fold": fold_id, "session_id": groups_te,
                                  "true_lap_time_s": y_te.values, "pred_lap_time_s": y_pred,
                                  "abs_error_s": np.abs(y_pred - y_te.values), "Signed_error_s": y_pred - y_te.values})
            err_df = pd.concat([err_df, meta_te.reset_index(drop=True)], axis=1)
            all_errors.append(err_df)

    results_df = pd.DataFrame(all_results)
    errors_df = pd.concat(all_errors, axis=0).reset_index(drop=True)
    print("\n=== Aggregate fold metrics ===")
    print(results_df.groupby("model")[["MAE", "RMSE", "R2", "train_time_s", "infer_time_s"]].agg(["mean", "std"]))
    if save_plots:
        results_df.to_csv("classical_fold_metrics.csv", index=False)
        errors_df.to_csv("classical_errors_per_lap.csv", index=False)

    if save_plots:
        sns.set_theme(style="whitegrid")
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=errors_df, x="model", y="abs_error_s")
        plt.ylabel("Absolute error [s]")
        plt.title("Lap time prediction absolute error by model")
        plt.tight_layout()
        plt.savefig("classical_error_distribution.png", dpi=150, bbox_inches='tight')
        plt.show()

    return results_df, errors_df

print("Classical ML functions loaded")

Classical ML functions loaded


### Quantum ML Functions

In [5]:
def build_qiskit_regressor(n_features: int, feature_map_reps: int = 1, ansatz_reps: int = 2,
                           maxiter: int = 100, optimizer: str = "COBYLA", use_gpu: bool = False):
    """Build a Qiskit quantum neural network regressor."""
    num_qubits = n_features
    feature_map = ZZFeatureMap(
        feature_dimension=num_qubits,
        reps=feature_map_reps,
        entanglement="linear"
    )
    ansatz = RealAmplitudes(
        num_qubits,
        reps=ansatz_reps,
        entanglement="linear"
    )
    qc = QuantumCircuit(num_qubits)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)

    # Configure Estimator - handle different qiskit API versions
    # Newer versions have different Estimator APIs that require observables in run()
    estimator = None
    estimator_error = None

    # Try to get qiskit version for compatibility handling
    try:
        import qiskit
        qiskit_version = tuple(map(int, qiskit.__version__.split('.')))
    except:
        qiskit_version = (0, 0, 0)

    if use_gpu:
        try:
            from qiskit_aer import AerSimulator
            simulator = AerSimulator(method='statevector')
            try:
                from qiskit_aer.primitives import Estimator as AerEstimator
                # Try with backend parameter (older API)
                try:
                    estimator = AerEstimator(backend=simulator)
                    print("  Using Aer quantum simulator (GPU acceleration if available)")
                except (TypeError, AttributeError):
                    # Newer API might not accept backend, or needs different setup
                    try:
                        estimator = AerEstimator()
                        print("  Using Aer Estimator (GPU acceleration if available)")
                    except Exception as e:
                        estimator_error = e
            except (ImportError, AttributeError) as e:
                estimator_error = e
        except (ImportError, Exception) as e:
            estimator_error = e

    # If GPU path didn't work, try standard Estimator
    if estimator is None:
        if Estimator is not None:
            try:
                estimator = Estimator()
                if estimator_error:
                    print(f"  GPU acceleration not available: {estimator_error}, using standard Estimator")
                else:
                    print("  Using standard Estimator")
            except Exception as e:
                estimator_error = e
                estimator = None

    # Final fallback: try Aer Estimator without backend
    if estimator is None:
        try:
            from qiskit_aer.primitives import Estimator as AerEstimator
            estimator = AerEstimator()
            print("  Using Aer Estimator as fallback")
        except ImportError:
            if estimator_error:
                raise ImportError(f"Could not import any Estimator. Errors: {estimator_error}. Please install qiskit-aer: pip install qiskit-aer") from estimator_error
            else:
                raise ImportError("Could not import any Estimator. Please install qiskit-aer: pip install qiskit-aer")

    obs_str = "Z" + ("I" * (num_qubits - 1)) if num_qubits > 1 else "Z"
    obs = SparsePauliOp(obs_str)

    # Prefer StatevectorEstimator for better EstimatorQNN compatibility
    # StatevectorEstimator is designed to work seamlessly with EstimatorQNN
    original_estimator = estimator
    try:
        from qiskit.primitives import StatevectorEstimator
        estimator = StatevectorEstimator()
        print("  Using StatevectorEstimator (recommended for EstimatorQNN compatibility)")
    except ImportError:
        # StatevectorEstimator not available, use original estimator
        estimator = original_estimator
        print("  Note: StatevectorEstimator not available, using alternative Estimator")

    # Construct EstimatorQNN - handle different qiskit API versions
    # The Estimator API changed significantly - newer versions require observables in run()
    # EstimatorQNN should handle this internally, but API changes can cause issues
    qnn = None
    qnn_error = None

    # Method 1: Standard approach with observables as SparsePauliOp
    try:
        qnn = EstimatorQNN(
            circuit=qc,
            estimator=estimator,
            input_params=feature_map.parameters,
            weight_params=ansatz.parameters,
            observables=obs
        )
    except (TypeError, AttributeError, RuntimeError, ValueError) as e:
        qnn_error = e
        error_msg = str(e).lower()

        # Method 2: Try with observables as list (some versions require list)
        if "observables" in error_msg or "run()" in error_msg or "missing" in error_msg or "required" in error_msg:
            try:
                qnn = EstimatorQNN(
                    circuit=qc,
                    estimator=estimator,
                    input_params=feature_map.parameters,
                    weight_params=ansatz.parameters,
                    observables=[obs]  # Try as list
                )
            except Exception as e2:
                qnn_error = e2
                # Method 3: Try without observables parameter (let QNN create default)
                # Some versions create observables internally from circuit
                try:
                    qnn = EstimatorQNN(
                        circuit=qc,
                        estimator=estimator,
                        input_params=feature_map.parameters,
                        weight_params=ansatz.parameters
                    )
                    # Try to set observables after creation if supported
                    if hasattr(qnn, 'observables'):
                        try:
                            qnn.observables = obs
                        except:
                            # If setting fails, try as list
                            try:
                                qnn.observables = [obs]
                            except:
                                pass
                except Exception as e3:
                    # Method 4: Try with StatevectorEstimator (alternative API)
                    try:
                        from qiskit.primitives import StatevectorEstimator
                        statevector_estimator = StatevectorEstimator()
                        qnn = EstimatorQNN(
                            circuit=qc,
                            estimator=statevector_estimator,
                            input_params=feature_map.parameters,
                            weight_params=ansatz.parameters,
                            observables=obs
                        )
                        print("  Using StatevectorEstimator as alternative")
                    except Exception as e4:
                        # All methods failed - provide detailed error with version info
                        try:
                            import qiskit
                            qiskit_ver = qiskit.__version__
                        except:
                            qiskit_ver = "unknown"
                        try:
                            import qiskit_machine_learning
                            qml_ver = qiskit_machine_learning.__version__
                        except:
                            qml_ver = "unknown"

                        raise RuntimeError(
                            f"Failed to create EstimatorQNN. Tried multiple approaches.\n"
                            f"  Method 1 (SparsePauliOp): {e}\n"
                            f"  Method 2 (list): {e2}\n"
                            f"  Method 3 (no observables): {e3}\n"
                            f"  Method 4 (StatevectorEstimator): {e4}\n"
                            f"\nDetected versions:\n"
                            f"  qiskit: {qiskit_ver}\n"
                            f"  qiskit-machine-learning: {qml_ver}\n"
                            f"\nThis is a qiskit version compatibility issue.\n"
                            f"Recommended fix:\n"
                            f"  pip install --upgrade qiskit-machine-learning>=0.7.0 qiskit-aer>=0.13.0\n"
                            f"Or try specific versions:\n"
                            f"  pip install qiskit-machine-learning==0.7.0 qiskit-aer==0.13.0"
                        ) from e4
        else:
            # Error not related to observables, re-raise
            raise

    if qnn is None:
        raise RuntimeError(f"Could not create EstimatorQNN. Error: {qnn_error}")
    iteration_count = [0]
    print_freq = max(1, maxiter // 10) if maxiter > 10 else maxiter

    def callback(weights, obj_func_eval):
        iteration_count[0] += 1
        if iteration_count[0] % print_freq == 0 or iteration_count[0] == maxiter:
            print(f"  Iteration {iteration_count[0]}/{maxiter} - Loss: {obj_func_eval:.6f}")
        return False

    # Configure optimizer with maxiter
    # In newer qiskit versions, optimizer must be instantiated, not passed as string
    try:
        from qiskit_algorithms.optimizers import COBYLA, SPSA, L_BFGS_B
        optimizer_map = {
            "COBYLA": COBYLA,
            "SPSA": SPSA,
            "L_BFGS_B": L_BFGS_B
        }
        opt_class = optimizer_map.get(optimizer.upper(), COBYLA)
        opt = opt_class(maxiter=maxiter)
    except ImportError:
        # Fallback for older qiskit versions
        try:
            from qiskit.algorithms.optimizers import COBYLA
            opt = COBYLA(maxiter=maxiter)
        except ImportError:
            # If all else fails, use COBYLA from qiskit-algorithms
            try:
                from qiskit_algorithms import COBYLA
                opt = COBYLA(maxiter=maxiter)
            except:
                # Last resort: pass as string (may not work but won't crash)
                opt = optimizer

    # Create regressor - maxiter is set on optimizer, not regressor
    regressor = NeuralNetworkRegressor(
        neural_network=qnn,
        optimizer=opt,
        initial_point=np.random.randn(qnn.num_weights) * 0.1,
        callback=callback
    )

    return regressor

def evaluate_quantum_models(years=None, session_name="R", cache_dir="cache",
                           max_events_per_year=None, quantum_n_components=4, max_samples=600,
                           n_splits=5, maxiter=100, use_gpu=False, feature_map_reps=1, ansatz_reps=2,
                           min_laps_per_driver=5):
    """Evaluate quantum ML model (Qiskit QNN) using group-based cross-validation."""
    if years is None:
        from datetime import datetime
        years = list(range(2020, min(2025, datetime.now().year + 1)))

    X_classical, X_quantum, y, meta, scalers, groups = load_sessions_and_build_dataset(
        years=years, session_name=session_name, cache_dir=cache_dir,
        max_events_per_year=max_events_per_year, quantum_n_components=quantum_n_components,
        min_laps_per_driver=min_laps_per_driver)

    if len(X_quantum) > max_samples:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(X_quantum), size=max_samples, replace=False)
        idx = np.sort(idx)
        X_quantum, y = X_quantum[idx], y.iloc[idx].reset_index(drop=True)
        meta, groups = meta.iloc[idx].reset_index(drop=True), groups.iloc[idx].reset_index(drop=True)

    print(f"Evaluating quantum model: {len(X_quantum)} samples, {X_quantum.shape[1]} features, {n_splits} CV folds, {maxiter} iterations")

    all_results, all_errors = [], []
    gkf = GroupKFold(n_splits=n_splits)
    splits = list(gkf.split(X_quantum, y, groups))

    for fold_id, (train_idx, test_idx) in enumerate(tqdm(splits, desc="  Quantum folds"), 1):
        X_train, X_test = X_quantum[train_idx], X_quantum[test_idx]
        y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)
        meta_test, groups_test = meta.iloc[test_idx].reset_index(drop=True), groups.iloc[test_idx].reset_index(drop=True)

        y_mean, y_std = y_train.mean(), y_train.std()
        y_train_scaled = np.clip((y_train - y_mean) / (3 * y_std), -1.0, 1.0)

        n_features = X_train.shape[1]
        q_regressor = build_qiskit_regressor(
            n_features,
            maxiter=maxiter,
            use_gpu=use_gpu,
            feature_map_reps=feature_map_reps,
            ansatz_reps=ansatz_reps
        )

        print(f"  Fold {fold_id}: Training...")
        t0 = time.perf_counter()
        q_regressor.fit(X_train, y_train_scaled.values)
        train_time = time.perf_counter() - t0
        t0 = time.perf_counter()
        y_pred_scaled = q_regressor.predict(X_test)
        infer_time = time.perf_counter() - t0

        # Ensure y_pred_scaled is 1D array
        y_pred_scaled = np.asarray(y_pred_scaled).flatten()

        y_pred = y_pred_scaled * (3 * y_std) + y_mean
        # Ensure y_pred is also 1D
        y_pred = np.asarray(y_pred).flatten()

        # Ensure y_test.values is 1D
        y_test_vals = np.asarray(y_test.values).flatten()

        mae = mean_absolute_error(y_test_vals, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test_vals, y_pred))  # Calculate RMSE manually for compatibility
        r2 = r2_score(y_test_vals, y_pred)

        all_results.append({"model": "qiskit_qnn", "fold": fold_id, "MAE": mae, "RMSE": rmse, "R2": r2,
                           "train_time_s": train_time, "infer_time_s": infer_time, "n_train": len(X_train), "n_test": len(X_test)})

        # Ensure all arrays are 1D for DataFrame creation
        err_df = pd.DataFrame({
            "model": "qiskit_qnn",
            "fold": fold_id,
            "session_id": np.asarray(groups_test).flatten(),
            "true_lap_time_s": y_test_vals,
            "pred_lap_time_s": y_pred,
            "abs_error_s": np.abs(y_pred - y_test_vals),
            "Signed_error_s": y_pred - y_test_vals
        })
        err_df = pd.concat([err_df.reset_index(drop=True), meta_test.reset_index(drop=True)], axis=1)
        all_errors.append(err_df)

    results_df = pd.DataFrame(all_results)
    errors_df = pd.concat(all_errors, axis=0).reset_index(drop=True)
    print("\n=== Aggregate fold metrics ===")
    agg_cols = ["MAE", "RMSE", "R2", "train_time_s", "infer_time_s"]
    print(results_df.groupby("model")[agg_cols].agg(["mean", "std"]))
    # Disabled intermediate CSV outputs for A1: appendix tables are generated later from in-memory fold results.
    # results_df.to_csv("quantum_fold_metrics.csv", index=False)
    # errors_df.to_csv("quantum_errors_per_lap.csv", index=False)
    return results_df, errors_df

print("Quantum ML functions loaded")

Quantum ML functions loaded


### Configuration and Comparison Functions

In [6]:
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from typing import List, Optional, Tuple, Dict, Any

IN_COLAB = 'google.colab' in sys.modules

TESTING_MODE = True

CURRENT_YEAR = datetime.now().year
FASTF1_YEARS = list(range(2020, min(2026, CURRENT_YEAR + 1))) if CURRENT_YEAR >= 2020 else list(range(2020, CURRENT_YEAR + 1))

# F1 Races per Year Configuration
# Note: From 2022, F1 introduced major regulation changes:
# - 13-inch wheels → 18-inch wheels (larger, heavier)
# - Increased car weight (minimum weight increased)
# - Ground effect aerodynamics reintroduced
# These changes significantly affect lap times and car performance characteristics
def get_max_events_per_year(year: int) -> Optional[int]:
    """Get maximum events per year based on F1 calendar."""
    if year == 2020:
        return 17  # COVID-19 shortened season
    elif year in [2021, 2022, 2023]:
        return 22  # Standard calendar
    elif year in [2024, 2025]:
        return 24  # Expanded calendar
    else:
        return None  # Use all available events

def get_max_events_for_years(years: List[int]) -> Optional[int]:
    """Get max events per year for a list of years."""
    max_events = [get_max_events_per_year(y) for y in years]
    if len(set(max_events)) == 1:
        return max_events[0]
    return None  # Different limits per year - handle in load function

TEST_CONFIG = {
    'years': [2022],
    'max_events_per_year': 2,
    'session_name': "R",
    'hgb_max_iter': 15,
    'n_splits': 2,
    'quantum_n_components': 4,
    'max_samples': 200,
    'quantum_maxiter': 10,
    'feature_map_reps': 1,
    'ansatz_reps': 1,
    'use_gpu': False
}

# MAXIMIZED FULL MODE CONFIGURATION
# Optimized for maximum performance and comprehensive evaluation
COLAB_CONFIG = {
    'years': FASTF1_YEARS,
    'max_events_per_year': None,  # Will use per-year limits (17/22/24)
    'session_name': "R",
    'hgb_max_iter': 1500,  # Increased from 1000 for better convergence
    'n_splits': 5,
    'quantum_n_components': 8,  # Increased from 4 for better expressiveness
    'max_samples': 5000,  # Increased from 1000 for better statistical power
    'quantum_maxiter': 200,  # Increased from 100 for better optimization
    'feature_map_reps': 2,  # Increased from 1 for better feature encoding
    'ansatz_reps': 3,  # Increased from 2 for deeper circuits
    'use_gpu': True
}

# MAXIMIZED STANDARD CONFIGURATION (for local/standard environments)
STANDARD_CONFIG = {
    'years': FASTF1_YEARS,
    'max_events_per_year': None,  # Will use per-year limits (17/22/24)
    'session_name': "R",
    'hgb_max_iter': 1000,  # Increased from 300
    'n_splits': 5,
    'quantum_n_components': 6,  # Increased from 4
    'max_samples': 2000,  # Increased from 600
    'quantum_maxiter': 150,  # Increased from 100
    'feature_map_reps': 2,  # Increased from 1
    'ansatz_reps': 3,  # Increased from 2
    'use_gpu': False
}

def validate_config(config: Dict[str, Any]) -> Dict[str, Any]:
    """Validate and sanitize configuration parameters."""
    validated = config.copy()

    if 'years' in validated:
        years = validated['years']
        if not isinstance(years, list) or len(years) == 0:
            raise ValueError("years must be a non-empty list")
        if not all(isinstance(y, int) and 2020 <= y <= min(2025, datetime.now().year) for y in years):
            raise ValueError(f"years must be integers between 2020 and {min(2025, datetime.now().year)}")

    if 'n_splits' in validated:
        if not isinstance(validated['n_splits'], int) or validated['n_splits'] < 2:
            raise ValueError("n_splits must be an integer >= 2")

    if 'quantum_n_components' in validated:
        if not isinstance(validated['quantum_n_components'], int) or validated['quantum_n_components'] < 1:
            raise ValueError("quantum_n_components must be an integer >= 1")

    if 'max_samples' in validated:
        if not isinstance(validated['max_samples'], int) or validated['max_samples'] < 1:
            raise ValueError("max_samples must be an integer >= 1")

    if 'hgb_max_iter' in validated:
        if not isinstance(validated['hgb_max_iter'], int) or validated['hgb_max_iter'] < 1:
            raise ValueError("hgb_max_iter must be an integer >= 1")

    if 'quantum_maxiter' in validated:
        if not isinstance(validated['quantum_maxiter'], int) or validated['quantum_maxiter'] < 1:
            raise ValueError("quantum_maxiter must be an integer >= 1")

    return validated

def get_config():
    """Get configuration based on environment (Colab, testing, or standard)."""
    if TESTING_MODE:
        print("TESTING MODE ENABLED - Using reduced iterations (10-20) for quick testing")
        print("   Set TESTING_MODE = False for full training")
        config = TEST_CONFIG.copy()
    elif IN_COLAB:
        print("Running in Google Colab - using A100 High RAM optimized settings")
        config = COLAB_CONFIG.copy()
    else:
        print("Running in standard environment - using default settings")
        config = STANDARD_CONFIG.copy()

    print(f"  Year range: {min(config['years'])}-{max(config['years'])} ({len(config['years'])} years)")

    return validate_config(config)

def check_resources():
    """Check and display system resources (CPU, RAM, GPU)."""
    try:
        import psutil
        import platform
        resources = {
            'platform': platform.system(),
            'cpu_count': psutil.cpu_count(),
            'memory_total_gb': psutil.virtual_memory().total / (1024**3),
            'memory_available_gb': psutil.virtual_memory().available / (1024**3),
            'in_colab': IN_COLAB
        }
        try:
            import torch
            if torch.cuda.is_available():
                resources['gpu_available'] = True
                resources['gpu_name'] = torch.cuda.get_device_name(0)
                resources['gpu_memory_gb'] = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            else:
                resources['gpu_available'] = False
        except ImportError:
            resources['gpu_available'] = False

        print("\n=== System Resources ===")
        print(f"Platform: {resources['platform']}")
        print(f"CPU Cores: {resources['cpu_count']}")
        print(f"Total RAM: {resources['memory_total_gb']:.2f} GB")
        print(f"Available RAM: {resources['memory_available_gb']:.2f} GB")
        if resources.get('gpu_available'):
            print(f"GPU: {resources.get('gpu_name', 'Unknown')}")
            print(f"GPU Memory: {resources.get('gpu_memory_gb', 0):.2f} GB")
        print("=" * 30)
        return resources
    except ImportError:
        print("Note: Install psutil for resource checking")
        return {}

def compare_all_models(years=None, session_name="R", cache_dir="cache",
                      max_events_per_year=None, quantum_n_components=4, quantum_max_samples=600,
                      n_splits=5, save_plots=True, quantum_maxiter=100, hgb_max_iter=300,
                      use_gpu=False, quantum_feature_map_reps=1,
                      quantum_ansatz_reps=2, min_laps_per_driver=5) -> Tuple[pd.DataFrame, dict]:
    """Run full comparison between classical and quantum ML models."""
    if years is None:
        years = FASTF1_YEARS

    # Step 1: Evaluate Classical Models
    print("Step 1: Evaluating Classical Models")
    try:
        classical_results, classical_errors = evaluate_classical_models(
            years=years,
            session_name=session_name,
            cache_dir=cache_dir,
            max_events_per_year=max_events_per_year,
            quantum_n_components=quantum_n_components,
            n_splits=n_splits,
            save_plots=False,  # Don't save plots here, save combined plots later
            hgb_max_iter=hgb_max_iter,
            min_laps_per_driver=min_laps_per_driver,
            max_samples=quantum_max_samples  # Apply same sample limit for fair comparison
        )
        if len(classical_results) == 0:
            raise RuntimeError("Classical model evaluation produced no results")
    except Exception as e:
        raise RuntimeError(f"Classical model evaluation failed: {e}") from e

    # Step 2: Evaluate Quantum Models
    print("\nStep 2: Evaluating Quantum Models")
    try:
        quantum_results, quantum_errors = evaluate_quantum_models(
            years=years,
            session_name=session_name,
            cache_dir=cache_dir,
            max_events_per_year=max_events_per_year,
            quantum_n_components=quantum_n_components,
            max_samples=quantum_max_samples,
            n_splits=n_splits,
            maxiter=quantum_maxiter,
            use_gpu=use_gpu,
            feature_map_reps=quantum_feature_map_reps,
            ansatz_reps=quantum_ansatz_reps,
            min_laps_per_driver=min_laps_per_driver
        )
        if len(quantum_results) == 0:
            raise RuntimeError("Quantum model evaluation produced no results")
    except Exception as e:
        raise RuntimeError(f"Quantum model evaluation failed: {e}") from e

    # Step 3: Generate Comparison Report
    print("\nStep 3: Generating Comparison Report")

    # Combine results
    all_results = pd.concat([classical_results, quantum_results], ignore_index=True)
    if len(all_results) == 0:
        raise RuntimeError("No results to compare - both model evaluations failed")

    # Create comparison summary
    comparison_df = all_results.groupby("model").agg({
        "MAE": ["mean", "std"],
        "RMSE": ["mean", "std"],
        "R2": ["mean", "std"],
        "train_time_s": ["mean", "std"],
        "infer_time_s": ["mean", "std"]
    }).round(4)

    print("\nComparison Summary:")
    print(comparison_df)

    # Statistical significance testing
    print("\n=== Statistical Significance Testing ===")
    try:
        from scipy.stats import ttest_rel, wilcoxon

        # Get fold-level metrics for each model
        classical_mae = classical_results.groupby('fold')['MAE'].mean()
        quantum_mae = quantum_results.groupby('fold')['MAE'].mean()

        # Ensure same number of folds for comparison
        if len(classical_mae) == len(quantum_mae):
            # Paired t-test for MAE
            t_stat, p_value_mae = ttest_rel(classical_mae, quantum_mae)
            print(f"\nMAE Comparison (Paired t-test):")
            print(f"  Classical mean: {classical_mae.mean():.4f} ± {classical_mae.std():.4f}")
            print(f"  Quantum mean: {quantum_mae.mean():.4f} ± {quantum_mae.std():.4f}")
            print(f"  t-statistic: {t_stat:.4f}, p-value: {p_value_mae:.4f}")
            if p_value_mae < 0.05:
                print(f"Significant difference (p < 0.05)")
            else:
                print(f"  No significant difference (p >= 0.05)")

            # Wilcoxon signed-rank test (non-parametric alternative)
            try:
                w_stat, p_value_wilcoxon = wilcoxon(classical_mae, quantum_mae)
                print(f"\nMAE Comparison (Wilcoxon signed-rank test):")
                print(f"  W-statistic: {w_stat:.4f}, p-value: {p_value_wilcoxon:.4f}")
                if p_value_wilcoxon < 0.05:
                    print(f"Significant difference (p < 0.05)")
                else:
                    print(f"  No significant difference (p >= 0.05)")
            except ValueError as e:
                print(f"  Note: Wilcoxon test not applicable: {e}")

            # R² comparison
            classical_r2 = classical_results.groupby('fold')['R2'].mean()
            quantum_r2 = quantum_results.groupby('fold')['R2'].mean()
            if len(classical_r2) == len(quantum_r2):
                t_stat_r2, p_value_r2 = ttest_rel(classical_r2, quantum_r2)
                print(f"\nR² Comparison (Paired t-test):")
                print(f"  Classical mean: {classical_r2.mean():.4f} ± {classical_r2.std():.4f}")
                print(f"  Quantum mean: {quantum_r2.mean():.4f} ± {quantum_r2.std():.4f}")
                print(f"  t-statistic: {t_stat_r2:.4f}, p-value: {p_value_r2:.4f}")
                if p_value_r2 < 0.05:
                    print(f"Significant difference (p < 0.05)")
                else:
                    print(f"  No significant difference (p >= 0.05)")
        else:
            print("  Note: Different number of folds, skipping statistical tests")
    except ImportError:
        print("  Note: scipy not available for statistical testing")
    except Exception as e:
        print(f"  Note: Statistical testing failed: {e}")

    # Combine errors for saving and plotting
    all_errors = pd.concat([classical_errors, quantum_errors], ignore_index=True)

    # Save results (disabled by default for A1 minimal-output workflow)
    if save_plots:
        try:
            comparison_df.to_csv("model_comparison_summary.csv")
            all_results.to_csv("all_models_detailed_results.csv", index=False)
            all_errors.to_csv("all_models_errors_per_lap.csv", index=False)
            print("\nResults saved to CSV files")
        except Exception as e:
            print(f"Warning: Could not save results to CSV: {e}")

    # Generate plots if requested
    if save_plots:
        try:
            sns.set_theme(style="whitegrid")
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))

            sns.boxplot(data=all_results, x="model", y="MAE", ax=axes[0, 0])
            axes[0, 0].set_title("Mean Absolute Error by Model")
            axes[0, 0].set_ylabel("MAE [s]")

            sns.boxplot(data=all_results, x="model", y="RMSE", ax=axes[0, 1])
            axes[0, 1].set_title("Root Mean Squared Error by Model")
            axes[0, 1].set_ylabel("RMSE [s]")

            sns.boxplot(data=all_results, x="model", y="R2", ax=axes[1, 0])
            axes[1, 0].set_title("R² Score by Model")
            axes[1, 0].set_ylabel("R²")

            sns.boxplot(data=all_results, x="model", y="train_time_s", ax=axes[1, 1])
            axes[1, 1].set_title("Training Time by Model")
            axes[1, 1].set_ylabel("Time [s]")
            axes[1, 1].set_yscale('log')

            plt.tight_layout()
            plt.savefig("model_comparison.png", dpi=150, bbox_inches='tight')
            plt.show()

            plt.figure(figsize=(10, 6))
            sns.boxplot(data=all_errors, x="model", y="abs_error_s")
            plt.ylabel("Absolute Error [s]")
            plt.title("Absolute Error Distribution: Classical vs Quantum Models")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig("error_distribution_comparison.png", dpi=150, bbox_inches='tight')
            plt.show()
            print("Comparison plots saved")
        except Exception as e:
            print(f"Warning: Could not generate plots: {e}")

    # Compile statistical test results
    stats_results = {}
    try:
        from scipy.stats import ttest_rel, wilcoxon
        classical_mae = classical_results.groupby('fold')['MAE'].mean()
        quantum_mae = quantum_results.groupby('fold')['MAE'].mean()
        if len(classical_mae) == len(quantum_mae):
            t_stat, p_value_mae = ttest_rel(classical_mae, quantum_mae)
            stats_results['mae_ttest'] = {'t_statistic': t_stat, 'p_value': p_value_mae}
            try:
                w_stat, p_value_wilcoxon = wilcoxon(classical_mae, quantum_mae)
                stats_results['mae_wilcoxon'] = {'w_statistic': w_stat, 'p_value': p_value_wilcoxon}
            except:
                pass
    except:
        pass

    return comparison_df, {
        "classical": {"results": classical_results, "errors": classical_errors},
        "quantum": {"results": quantum_results, "errors": quantum_errors},
        "statistical_tests": stats_results
    }

def create_data_exploration_plots(X_classical, y, meta, save_dir="."):
    """Create data exploration visualizations focused on understanding the dataset"""
    print("\n=== Creating Data Exploration Visualizations ===")
    sns.set_theme(style="whitegrid")

    # Reduced to 2x2 grid - focus on most relevant plots for F1 engineers
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Plot 1: Lap Time Distribution (Top Left)
    axes[0, 0].hist(y.values, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0, 0].set_xlabel('Lap Time [s]')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution of Lap Times')
    axes[0, 0].axvline(y.mean(), color='r', linestyle='--', linewidth=2, label=f'Mean: {y.mean():.2f}s')
    axes[0, 0].axvline(y.median(), color='g', linestyle='--', linewidth=2, label=f'Median: {y.median():.2f}s')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    meta_with_y = meta.copy()
    meta_with_y['LapTime_sec'] = y.values

    # Plot 2: Tyre Compound Effects (Top Right) - Critical for strategy
    if 'Compound' in meta.columns:
        compound_data = meta_with_y[['Compound', 'LapTime_sec']].dropna()
        if len(compound_data) > 0:
            sns.boxplot(data=compound_data, x='Compound', y='LapTime_sec', ax=axes[0, 1], palette='Set2')
            axes[0, 1].set_ylabel('Lap Time [s]')
            axes[0, 1].set_xlabel('Tyre Compound')
            axes[0, 1].set_title('Lap Time Distribution by Tyre Compound')
            axes[0, 1].tick_params(axis='x', rotation=45)
            axes[0, 1].grid(True, alpha=0.3, axis='y')
        else:
            axes[0, 1].text(0.5, 0.5, 'No tyre compound data available',
                          ha='center', va='center', transform=axes[0, 1].transAxes, fontsize=12)
            axes[0, 1].set_title('Lap Time Distribution by Tyre Compound (No Data)')
    else:
        axes[0, 1].text(0.5, 0.5, 'Tyre compound data not available',
                      ha='center', va='center', transform=axes[0, 1].transAxes, fontsize=12)
        axes[0, 1].set_title('Lap Time Distribution by Tyre Compound (No Data)')

    # Plot 3: Tyre Life Impact (Bottom Left) - Critical for degradation modeling
    if 'TyreLife' in meta.columns:
        tyre_data = meta_with_y[['TyreLife', 'LapTime_sec']].dropna()
        if len(tyre_data) > 0:
            if len(tyre_data) > 5000:
                tyre_data = tyre_data.sample(n=5000, random_state=42)
            axes[1, 0].scatter(tyre_data['TyreLife'], tyre_data['LapTime_sec'],
                             alpha=0.3, s=10, color='darkblue')
            axes[1, 0].set_xlabel('Tyre Life (Laps)')
            axes[1, 0].set_ylabel('Lap Time [s]')
            axes[1, 0].set_title('Lap Time vs Tyre Life (Degradation Effect)')
            axes[1, 0].grid(True, alpha=0.3)
            if len(tyre_data) > 1:
                z = np.polyfit(tyre_data['TyreLife'], tyre_data['LapTime_sec'], 1)
                p = np.poly1d(z)
                axes[1, 0].plot(tyre_data['TyreLife'], p(tyre_data['TyreLife']),
                              "r--", alpha=0.8, linewidth=2, label=f'Trend: +{z[0]:.3f}s/lap')
                axes[1, 0].legend()
        else:
            axes[1, 0].text(0.5, 0.5, 'No tyre life data available',
                          ha='center', va='center', transform=axes[1, 0].transAxes, fontsize=12)
            axes[1, 0].set_title('Lap Time vs Tyre Life (No Data)')
    else:
        axes[1, 0].text(0.5, 0.5, 'Tyre life data not available',
                      ha='center', va='center', transform=axes[1, 0].transAxes, fontsize=12)
        axes[1, 0].set_title('Lap Time vs Tyre Life (No Data)')

    # Plot 4: Feature Correlation Matrix / Association Matrix (Bottom Right)
    # This is critical for understanding feature relationships
    if len(X_classical.columns) > 1:
        numeric_cols = X_classical.select_dtypes(include=[np.number]).columns
        # Select top features by correlation with target
        if len(numeric_cols) > 0:
            try:
                # Calculate correlations with target
                target_corrs = []
                for col in numeric_cols:
                    try:
                        corr = abs(X_classical[col].corr(y))
                        if not np.isnan(corr):
                            target_corrs.append((col, corr))
                    except:
                        continue

                # Sort by correlation and take top 15-20 features
                target_corrs.sort(key=lambda x: x[1], reverse=True)
                top_features = [col for col, _ in target_corrs[:min(20, len(target_corrs))]]

                if len(top_features) > 1:
                    # Include target in correlation matrix
                    corr_data = pd.concat([X_classical[top_features], y], axis=1)
                    corr_matrix = corr_data.corr()

                    if not corr_matrix.empty and len(corr_matrix) > 1:
                        # Create heatmap with better formatting
                        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # Mask upper triangle
                        sns.heatmap(corr_matrix, annot=False, cmap='RdBu_r', center=0,
                                   square=True, ax=axes[1, 1], cbar_kws={"shrink": 0.8},
                                   mask=mask, vmin=-1, vmax=1, linewidths=0.5)
                        axes[1, 1].set_title(f'Feature Correlation Matrix (Top {len(top_features)} Features)')
                        # Rotate labels for readability
                        axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45, ha='right')
                        axes[1, 1].set_yticklabels(axes[1, 1].get_yticklabels(), rotation=0)
                    else:
                        axes[1, 1].text(0.5, 0.5, 'Insufficient data for correlation matrix',
                                      ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=12)
                        axes[1, 1].set_title('Feature Correlation Matrix (Error)')
                else:
                    axes[1, 1].text(0.5, 0.5, 'Insufficient features for correlation matrix',
                                  ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=12)
                    axes[1, 1].set_title('Feature Correlation Matrix (Error)')
            except Exception as e:
                axes[1, 1].text(0.5, 0.5, f'Could not generate correlation matrix:\n{str(e)[:50]}...',
                              ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=10)
                axes[1, 1].set_title('Feature Correlation Matrix (Error)')
        else:
            axes[1, 1].text(0.5, 0.5, 'No numeric features available',
                          ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=12)
            axes[1, 1].set_title('Feature Correlation Matrix (No Data)')
    else:
        axes[1, 1].text(0.5, 0.5, 'Insufficient features for correlation matrix',
                      ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=12)
        axes[1, 1].set_title('Feature Correlation Matrix (No Data)')

    plt.tight_layout()
    plt.savefig(f"{save_dir}/data_exploration.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Data exploration plots saved")

def create_prediction_visualizations(all_errors, save_dir="."):
    """Create prediction vs actual and residual plots for all models."""
    print("\n=== Creating Prediction Visualizations ===")
    sns.set_theme(style="whitegrid")

    models = all_errors['model'].unique()
    n_models = len(models)

    fig, axes = plt.subplots(n_models, 2, figsize=(14, 5*n_models))
    if n_models == 1:
        axes = axes.reshape(1, -1)

    for idx, model in enumerate(models):
        model_errors = all_errors[all_errors['model'] == model]

        if len(model_errors) > 5000:
            model_errors = model_errors.sample(n=5000, random_state=42)

        axes[idx, 0].scatter(model_errors['true_lap_time_s'], model_errors['pred_lap_time_s'],
                            alpha=0.3, s=10)
        min_val = min(model_errors['true_lap_time_s'].min(), model_errors['pred_lap_time_s'].min())
        max_val = max(model_errors['true_lap_time_s'].max(), model_errors['pred_lap_time_s'].max())
        axes[idx, 0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
        axes[idx, 0].set_xlabel('Actual Lap Time [s]')
        axes[idx, 0].set_ylabel('Predicted Lap Time [s]')
        axes[idx, 0].set_title(f'{model.upper()} - Prediction vs Actual')
        axes[idx, 0].legend()
        axes[idx, 0].grid(True, alpha=0.3)

        residuals = model_errors['pred_lap_time_s'] - model_errors['true_lap_time_s']
        axes[idx, 1].scatter(model_errors['true_lap_time_s'], residuals, alpha=0.3, s=10)
        axes[idx, 1].axhline(y=0, color='r', linestyle='--', lw=2)
        axes[idx, 1].set_xlabel('Actual Lap Time [s]')
        axes[idx, 1].set_ylabel('Residuals [s]')
        axes[idx, 1].set_title(f'{model.upper()} - Residuals Plot')
        axes[idx, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/prediction_analysis.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Prediction visualizations saved")

def create_error_analysis_plots(all_errors, save_dir="."):
    """Create error analysis plots by driver, compound, position, etc."""
    print("\n=== Creating Error Analysis Visualizations ===")
    sns.set_theme(style="whitegrid")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    if 'Driver' in all_errors.columns:
        driver_errors = all_errors.groupby(['model', 'Driver'])['abs_error_s'].mean().reset_index()
        top_drivers = all_errors['Driver'].value_counts().head(15).index
        driver_errors = driver_errors[driver_errors['Driver'].isin(top_drivers)]
        if len(driver_errors) > 0:
            sns.barplot(data=driver_errors, x='Driver', y='abs_error_s', hue='model', ax=axes[0, 0])
            axes[0, 0].set_ylabel('Mean Absolute Error [s]')
            axes[0, 0].set_title('Error by Driver (Top 15)')
            axes[0, 0].tick_params(axis='x', rotation=45)
            axes[0, 0].legend(title='Model')

    if 'Compound' in all_errors.columns:
        compound_errors = all_errors.groupby(['model', 'Compound'])['abs_error_s'].mean().reset_index()
        if len(compound_errors) > 0:
            sns.barplot(data=compound_errors, x='Compound', y='abs_error_s', hue='model', ax=axes[0, 1])
            axes[0, 1].set_ylabel('Mean Absolute Error [s]')
            axes[0, 1].set_title('Error by Tyre Compound')
            axes[0, 1].legend(title='Model')

    if 'Position' in all_errors.columns:
        position_errors = all_errors.groupby(['model', 'Position'])['abs_error_s'].mean().reset_index()
        position_errors['PositionBin'] = pd.cut(
            position_errors['Position'],
            bins=[0, 5, 10, 15, 20, 25],
            labels=['1-5', '6-10', '11-15', '16-20', '21+']
        )
        position_bin_errors = position_errors.groupby(['model', 'PositionBin'])['abs_error_s'].mean().reset_index()
        if len(position_bin_errors) > 0:
            sns.barplot(data=position_bin_errors, x='PositionBin', y='abs_error_s', hue='model', ax=axes[1, 0])
            axes[1, 0].set_xlabel('Race Position')
            axes[1, 0].set_ylabel('Mean Absolute Error [s]')
            axes[1, 0].set_title('Error by Race Position')
            axes[1, 0].legend(title='Model')

    for model in all_errors['model'].unique():
        model_errors = all_errors[all_errors['model'] == model]['abs_error_s']
        axes[1, 1].hist(model_errors, alpha=0.5, label=model, bins=30)
    axes[1, 1].set_xlabel('Absolute Error [s]')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Error Distribution Comparison')
    axes[1, 1].legend()

    plt.tight_layout()
    plt.savefig(f"{save_dir}/error_analysis.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Error analysis plots saved")

def create_performance_metrics_plots(all_results, save_dir="."):
    """Create comprehensive performance metrics visualizations."""
    print("\n=== Creating Performance Metrics Visualizations ===")
    sns.set_theme(style="whitegrid")

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    metrics_summary = all_results.groupby('model').agg({
        'MAE': 'mean',
        'RMSE': 'mean',
        'R2': 'mean'
    }).reset_index()

    x = np.arange(len(metrics_summary))
    width = 0.25
    axes[0, 0].bar(x - width, metrics_summary['MAE'], width, label='MAE', alpha=0.8)
    axes[0, 0].bar(x, metrics_summary['RMSE'], width, label='RMSE', alpha=0.8)
    axes[0, 0].bar(x + width, metrics_summary['R2'] * 10, width, label='R² (×10)', alpha=0.8)
    axes[0, 0].set_xlabel('Model')
    axes[0, 0].set_ylabel('Metric Value')
    axes[0, 0].set_title('Average Performance Metrics by Model')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(metrics_summary['model'])
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3, axis='y')

    for model in all_results['model'].unique():
        model_data = all_results[all_results['model'] == model]
        axes[0, 1].plot(model_data['fold'], model_data['MAE'], marker='o', label=model, alpha=0.7)
    axes[0, 1].set_xlabel('Fold')
    axes[0, 1].set_ylabel('MAE [s]')
    axes[0, 1].set_title('MAE Across Cross-Validation Folds')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    for model in all_results['model'].unique():
        model_data = all_results[all_results['model'] == model]
        axes[0, 2].plot(model_data['fold'], model_data['R2'], marker='o', label=model, alpha=0.7)
    axes[0, 2].set_xlabel('Fold')
    axes[0, 2].set_ylabel('R² Score')
    axes[0, 2].set_title('R² Score Across Cross-Validation Folds')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    sns.boxplot(data=all_results, x='model', y='train_time_s', ax=axes[1, 0])
    axes[1, 0].set_ylabel('Training Time [s]')
    axes[1, 0].set_title('Training Time Distribution by Model')
    axes[1, 0].set_yscale('log')
    axes[1, 0].tick_params(axis='x', rotation=45)

    sns.boxplot(data=all_results, x='model', y='infer_time_s', ax=axes[1, 1])
    axes[1, 1].set_ylabel('Inference Time [s]')
    axes[1, 1].set_title('Inference Time Distribution by Model')
    axes[1, 1].tick_params(axis='x', rotation=45)

    avg_perf = all_results.groupby('model').agg({
        'MAE': 'mean',
        'train_time_s': 'mean'
    }).reset_index()
    for _, row in avg_perf.iterrows():
        axes[1, 2].scatter(row['train_time_s'], row['MAE'], s=200, alpha=0.6, label=row['model'])
        axes[1, 2].annotate(
            row['model'],
            (row['train_time_s'], row['MAE']),
            xytext=(5, 5),
            textcoords='offset points'
        )
    axes[1, 2].set_xlabel('Training Time [s] (log scale)')
    axes[1, 2].set_ylabel('Mean Absolute Error [s]')
    axes[1, 2].set_title('Performance vs Training Time Trade-off')
    axes[1, 2].set_xscale('log')
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/performance_metrics.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Performance metrics plots saved")

def create_feature_importance_plot(X_classical, y, save_dir="."):
    """
    Create feature importance visualization using correlation with target.
    """
    print("\n=== Creating Feature Importance Visualization ===")
    sns.set_theme(style="whitegrid")

    correlations = []
    for col in X_classical.columns:
        if pd.api.types.is_numeric_dtype(X_classical[col]):
            try:
                corr = np.abs(X_classical[col].corr(y))
                if not np.isnan(corr):
                    correlations.append({'Feature': col, 'Abs_Correlation': corr})
            except Exception:
                continue

    if len(correlations) == 0:
        print("Warning: No valid correlations found. Skipping feature importance plot.")
        return

    corr_df = pd.DataFrame(correlations).sort_values('Abs_Correlation', ascending=False).head(20)

    if len(corr_df) == 0:
        print("Warning: No features to plot. Skipping feature importance plot.")
        return

    plt.figure(figsize=(10, 8))
    plt.barh(range(len(corr_df)), corr_df['Abs_Correlation'].values)
    plt.yticks(range(len(corr_df)), corr_df['Feature'].values)
    plt.xlabel('Absolute Correlation with Lap Time')
    plt.title('Top 20 Features by Correlation with Target')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"{save_dir}/feature_importance.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Feature importance plot saved")

def create_classical_vs_quantum_comparison(all_results, all_errors, save_dir="."):
    """Create comprehensive classical vs quantum comparison visualization."""
    print("\n=== Creating Classical vs Quantum Comparison Visualization ===")
    sns.set_theme(style="whitegrid")

    # Separate classical and quantum models
    classical_models = [m for m in all_results['model'].unique() if 'quantum' not in m.lower()]
    quantum_models = [m for m in all_results['model'].unique() if 'quantum' in m.lower()]

    if len(classical_models) == 0 or len(quantum_models) == 0:
        print("Warning: Need both classical and quantum models for comparison. Skipping.")
        return

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Classical vs Quantum ML: Decision Guide for F1 Engineers',
                 fontsize=16, fontweight='bold', y=0.995)

    # Plot 1: MAE Comparison (Top Left) - Most critical metric
    classical_data = all_results[all_results['model'].isin(classical_models)]
    quantum_data = all_results[all_results['model'].isin(quantum_models)]

    data_for_plot = pd.concat([
        pd.DataFrame({'Model_Type': 'Classical', 'MAE': classical_data['MAE']}),
        pd.DataFrame({'Model_Type': 'Quantum', 'MAE': quantum_data['MAE']})
    ])

    sns.boxplot(data=data_for_plot, x='Model_Type', y='MAE', ax=axes[0, 0], palette=['#2E86AB', '#A23B72'])
    axes[0, 0].set_ylabel('Mean Absolute Error [s]', fontsize=11)
    axes[0, 0].set_xlabel('Model Type', fontsize=11)
    axes[0, 0].set_title('Prediction Accuracy: Classical vs Quantum', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='y')

    # Add mean values as text
    classical_mean = classical_data['MAE'].mean()
    quantum_mean = quantum_data['MAE'].mean()
    axes[0, 0].text(0, classical_mean, f'{classical_mean:.3f}s',
                   ha='center', va='bottom', fontweight='bold', fontsize=10)
    axes[0, 0].text(1, quantum_mean, f'{quantum_mean:.3f}s',
                   ha='center', va='bottom', fontweight='bold', fontsize=10)

    # Plot 2: R² Comparison (Top Middle) - Variance explained
    data_r2 = pd.concat([
        pd.DataFrame({'Model_Type': 'Classical', 'R2': classical_data['R2']}),
        pd.DataFrame({'Model_Type': 'Quantum', 'R2': quantum_data['R2']})
    ])

    sns.boxplot(data=data_r2, x='Model_Type', y='R2', ax=axes[0, 1], palette=['#2E86AB', '#A23B72'])
    axes[0, 1].set_ylabel('R² Score', fontsize=11)
    axes[0, 1].set_xlabel('Model Type', fontsize=11)
    axes[0, 1].set_title('Variance Explained: Classical vs Quantum', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')

    classical_r2_mean = classical_data['R2'].mean()
    quantum_r2_mean = quantum_data['R2'].mean()
    axes[0, 1].text(0, classical_r2_mean, f'{classical_r2_mean:.3f}',
                   ha='center', va='bottom', fontweight='bold', fontsize=10)
    axes[0, 1].text(1, quantum_r2_mean, f'{quantum_r2_mean:.3f}',
                   ha='center', va='bottom', fontweight='bold', fontsize=10)

    # Plot 3: Training Time Comparison (Top Right) - Computational cost
    data_time = pd.concat([
        pd.DataFrame({'Model_Type': 'Classical', 'Time': classical_data['train_time_s']}),
        pd.DataFrame({'Model_Type': 'Quantum', 'Time': quantum_data['train_time_s']})
    ])

    sns.boxplot(data=data_time, x='Model_Type', y='Time', ax=axes[0, 2], palette=['#2E86AB', '#A23B72'])
    axes[0, 2].set_ylabel('Training Time [s]', fontsize=11)
    axes[0, 2].set_xlabel('Model Type', fontsize=11)
    axes[0, 2].set_title('Training Time: Classical vs Quantum', fontsize=12, fontweight='bold')
    axes[0, 2].set_yscale('log')
    axes[0, 2].grid(True, alpha=0.3, axis='y')

    # Plot 4: Error Distribution Comparison (Bottom Left)
    classical_errors = all_errors[all_errors['model'].isin(classical_models)]['abs_error_s']
    quantum_errors = all_errors[all_errors['model'].isin(quantum_models)]['abs_error_s']

    axes[1, 0].hist(classical_errors, bins=50, alpha=0.6, label='Classical', color='#2E86AB', density=True)
    axes[1, 0].hist(quantum_errors, bins=50, alpha=0.6, label='Quantum', color='#A23B72', density=True)
    axes[1, 0].set_xlabel('Absolute Error [s]', fontsize=11)
    axes[1, 0].set_ylabel('Density', fontsize=11)
    axes[1, 0].set_title('Error Distribution: Classical vs Quantum', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, axis='y')

    # Plot 5: Performance vs Time Trade-off (Bottom Middle)
    classical_perf = classical_data.groupby('model').agg({
        'MAE': 'mean',
        'train_time_s': 'mean'
    }).reset_index()
    quantum_perf = quantum_data.groupby('model').agg({
        'MAE': 'mean',
        'train_time_s': 'mean'
    }).reset_index()

    for _, row in classical_perf.iterrows():
        axes[1, 1].scatter(row['train_time_s'], row['MAE'], s=300, alpha=0.7,
                          color='#2E86AB', marker='o', edgecolors='black', linewidth=1.5)
        axes[1, 1].annotate('Classical', (row['train_time_s'], row['MAE']),
                           xytext=(10, 10), textcoords='offset points', fontsize=9, fontweight='bold')

    for _, row in quantum_perf.iterrows():
        axes[1, 1].scatter(row['train_time_s'], row['MAE'], s=300, alpha=0.7,
                          color='#A23B72', marker='s', edgecolors='black', linewidth=1.5)
        axes[1, 1].annotate('Quantum', (row['train_time_s'], row['MAE']),
                           xytext=(10, -15), textcoords='offset points', fontsize=9, fontweight='bold')

    axes[1, 1].set_xlabel('Training Time [s] (log scale)', fontsize=11)
    axes[1, 1].set_ylabel('Mean Absolute Error [s]', fontsize=11)
    axes[1, 1].set_title('Performance vs Training Time Trade-off', fontsize=12, fontweight='bold')
    axes[1, 1].set_xscale('log')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].invert_yaxis()  # Lower MAE is better, so invert

    # Plot 6: Summary Recommendation (Bottom Right)
    axes[1, 2].axis('off')

    # Calculate summary statistics
    classical_mae_mean = classical_data['MAE'].mean()
    quantum_mae_mean = quantum_data['MAE'].mean()
    classical_time_mean = classical_data['train_time_s'].mean()
    quantum_time_mean = quantum_data['train_time_s'].mean()

    # Determine recommendation
    mae_winner = "Classical" if classical_mae_mean < quantum_mae_mean else "Quantum"
    mae_diff = abs(classical_mae_mean - quantum_mae_mean)
    time_ratio = quantum_time_mean / classical_time_mean if classical_time_mean > 0 else float('inf')

    recommendation_text = f"""
    COMPARISON SUMMARY

    Accuracy (MAE):
    • Classical: {classical_mae_mean:.4f}s
    • Quantum: {quantum_mae_mean:.4f}s
    • Winner: {mae_winner} ({mae_diff:.4f}s better)

    Training Time:
    • Classical: {classical_time_mean:.1f}s
    • Quantum: {quantum_time_mean:.1f}s
    • Quantum is {time_ratio:.1f}x {'slower' if time_ratio > 1 else 'faster'}

    RECOMMENDATION:
    """

    if classical_mae_mean < quantum_mae_mean and time_ratio > 2:
        recommendation_text += "Use CLASSICAL ML\n• Better accuracy\n• Much faster training\n• Production-ready"
    elif quantum_mae_mean < classical_mae_mean and time_ratio < 3:
        recommendation_text += "Consider QUANTUM ML\n• Better accuracy\n• Acceptable training time\n• Research/experimental"
    else:
        recommendation_text += "Use CLASSICAL ML\n• Similar or better accuracy\n• Faster training\n• More practical for F1 teams"

    axes[1, 2].text(0.5, 0.5, recommendation_text, ha='center', va='center',
                   transform=axes[1, 2].transAxes, fontsize=11,
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                   family='monospace')

    plt.tight_layout()
    plt.savefig(f"{save_dir}/classical_vs_quantum_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("Classical vs Quantum comparison visualization saved")

def create_all_visualizations(X_classical, y, meta, all_results, all_errors, save_dir="."):
    """Create all visualizations in one call, focused on classical vs quantum comparison."""
    # Legacy helper (not used by the A1 minimal export workflow)
    print("Creating comprehensive visualizations...")

    # Data exploration (understanding the dataset)
    create_data_exploration_plots(X_classical, y, meta, save_dir)

    # KEY VISUALIZATION: Classical vs Quantum comparison
    create_classical_vs_quantum_comparison(all_results, all_errors, save_dir)

    # Detailed analysis visualizations
    create_prediction_visualizations(all_errors, save_dir)
    create_error_analysis_plots(all_errors, save_dir)
    create_performance_metrics_plots(all_results, save_dir)

    # Feature importance (less critical for comparison, but useful)
    create_feature_importance_plot(X_classical, y, save_dir)

    print("\n" + "=" * 80)
    print("ALL VISUALIZATIONS COMPLETE!")
    print("=" * 80)
    print("\nKEY OUTPUT: classical_vs_quantum_comparison.png")
    print("This visualization provides the decision guide for F1 engineers.")
    print("=" * 80)

print("Configuration and comparison functions loaded")
print("Visualization functions loaded")

Configuration and comparison functions loaded
Visualization functions loaded


In [7]:
### KPI Tracking and Domain-Specific Metrics

# KPI Tracking Functions
def track_kpis(func):
    """
    Decorator to track KPIs (Key Performance Indicators) for any function.
    Tracks execution time, memory usage, and optionally GPU utilization.

    Usage:
        @track_kpis
        def my_function():
            # function code
            return result
    """
    def wrapper(*args, **kwargs):
        import psutil
        import os
        import time as time_module

        process = psutil.Process(os.getpid())

        # Track start metrics
        start_time = time_module.perf_counter()
        start_memory_mb = process.memory_info().rss / (1024 * 1024)
        peak_memory_before = getattr(process.memory_info(), 'peak_wss', process.memory_info().rss) / (1024 * 1024)

        # Track GPU if available
        gpu_metrics = {}
        try:
            import GPUtil
            gpus = GPUtil.getGPUs()
            if gpus:
                gpu_metrics['gpu_utilization_before'] = gpus[0].load * 100
                gpu_metrics['gpu_memory_before_mb'] = gpus[0].memoryUsed
        except (ImportError, IndexError, AttributeError):
            pass

        # Execute function
        try:
            result = func(*args, **kwargs)
        finally:
            # Track end metrics
            end_time = time_module.perf_counter()
            end_memory_mb = process.memory_info().rss / (1024 * 1024)
            peak_memory_after = getattr(process.memory_info(), 'peak_wss', process.memory_info().rss) / (1024 * 1024)

            execution_time = end_time - start_time
            memory_delta_mb = end_memory_mb - start_memory_mb
            peak_memory_mb = max(peak_memory_after, peak_memory_before)

            # Track GPU after
            try:
                import GPUtil
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu_metrics['gpu_utilization_after'] = gpus[0].load * 100
                    gpu_metrics['gpu_memory_after_mb'] = gpus[0].memoryUsed
                    gpu_metrics['gpu_memory_delta_mb'] = gpus[0].memoryUsed - gpu_metrics.get('gpu_memory_before_mb', 0)
            except (ImportError, IndexError, AttributeError):
                pass

            # Store KPIs in function result if it's a dict, or attach as attribute
            kpis = {
                'execution_time_s': execution_time,
                'memory_start_mb': start_memory_mb,
                'memory_end_mb': end_memory_mb,
                'memory_delta_mb': memory_delta_mb,
                'peak_memory_mb': peak_memory_mb,
                **gpu_metrics
            }

            if isinstance(result, dict):
                result['_kpis'] = kpis
            else:
                # Attach as attribute if possible
                try:
                    result._kpis = kpis
                except:
                    pass

        return result
    return wrapper

def calculate_f1_domain_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                                sectors: Optional[np.ndarray] = None,
                                race_laps: int = 50) -> Dict[str, float]:
    """Calculate F1-specific domain metrics for lap time prediction."""
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    metrics = {}

    # Standard metrics (already calculated elsewhere, but included for completeness)
    metrics['mae_seconds'] = mean_absolute_error(y_true, y_pred)
    metrics['rmse_seconds'] = np.sqrt(mean_squared_error(y_true, y_pred))
    metrics['r2'] = r2_score(y_true, y_pred)

    # F1-specific metrics
    # 1. Mean Absolute Percentage Error (MAPE) - more intuitive than absolute error
    # Avoid division by zero
    valid_mask = y_true != 0
    if valid_mask.sum() > 0:
        metrics['mape_percent'] = np.mean(np.abs((y_true[valid_mask] - y_pred[valid_mask]) / y_true[valid_mask])) * 100
    else:
        metrics['mape_percent'] = np.nan

    # 2. Race-level error (total error over full race)
    # Critical for strategy: small per-lap errors compound over 50 laps
    metrics['race_error_50laps_seconds'] = metrics['mae_seconds'] * race_laps
    metrics['race_error_50laps_percent'] = metrics['mape_percent'] * race_laps / 100 if not np.isnan(metrics['mape_percent']) else np.nan

    # 3. Directional accuracy (overestimate vs underestimate)
    errors = y_pred - y_true
    metrics['overestimate_pct'] = (errors > 0).mean() * 100  # % of predictions that are too slow
    metrics['underestimate_pct'] = (errors < 0).mean() * 100  # % of predictions that are too fast
    metrics['exact_pct'] = (errors == 0).mean() * 100  # % of exact predictions (rare)

    # 4. Mean signed error (bias)
    metrics['mean_signed_error_seconds'] = errors.mean()
    metrics['median_signed_error_seconds'] = np.median(errors)

    # 5. Large error rate (errors > 1 second - critical for strategy)
    metrics['large_error_rate_pct'] = (np.abs(errors) > 1.0).mean() * 100

    # 6. Sector-specific errors (if sector data available)
    if sectors is not None:
        sectors = np.asarray(sectors).flatten()
        unique_sectors = np.unique(sectors)
        for sector in unique_sectors:
            sector_mask = sectors == sector
            if sector_mask.sum() > 0:
                metrics[f'mae_{sector.lower()}_seconds'] = mean_absolute_error(
                    y_true[sector_mask], y_pred[sector_mask]
                )
                metrics[f'rmse_{sector.lower()}_seconds'] = np.sqrt(
                    mean_squared_error(y_true[sector_mask], y_pred[sector_mask])
                )

    return metrics

def interpret_metrics_for_f1_teams(mae: float, rmse: float, r2: float,
                                   f1_metrics: Optional[Dict[str, float]] = None) -> Dict[str, str]:
    """Provide F1 team-friendly interpretation of metrics."""
    interpretations = {}

    # MAE interpretation
    race_error = mae * 50  # For 50-lap race
    if mae < 0.1:
        mae_quality = "excellent"
    elif mae < 0.5:
        mae_quality = "very good"
    elif mae < 1.0:
        mae_quality = "acceptable"
    elif mae < 2.0:
        mae_quality = "moderate"
    else:
        mae_quality = "poor"

    interpretations['mae_interpretation'] = (
        f"Average prediction error: {mae:.3f}s per lap. "
        f"For a 50-lap race: ±{race_error:.1f}s total error. "
        f"Quality: {mae_quality}. "
        f"F1 teams typically work with ±0.5s precision for strategy decisions."
    )

    # RMSE interpretation
    if rmse < 0.5:
        rmse_quality = "excellent"
    elif rmse < 1.0:
        rmse_quality = "good"
    elif rmse < 2.0:
        rmse_quality = "acceptable"
    else:
        rmse_quality = "poor"

    interpretations['rmse_interpretation'] = (
        f"RMSE of {rmse:.3f}s penalizes large errors more than MAE. "
        f"Quality: {rmse_quality}. "
        f"Critical for strategy: large errors (>1s) can lead to wrong pit stop timing decisions."
    )

    # R² interpretation
    variance_explained = r2 * 100 if r2 >= 0 else 0
    if r2 > 0.9:
        r2_quality = "excellent"
    elif r2 > 0.7:
        r2_quality = "good"
    elif r2 > 0.5:
        r2_quality = "moderate"
    elif r2 > 0:
        r2_quality = "poor"
    else:
        r2_quality = "very poor (worse than baseline)"

    interpretations['r2_interpretation'] = (
        f"R² of {r2:.3f} indicates model explains {variance_explained:.1f}% of variance. "
        f"Quality: {r2_quality}. "
        f"Note: R² relates to linear fit, but still useful as relative performance indicator. "
        f"Negative R² means model performs worse than predicting the mean."
    )

    # F1-specific interpretations
    if f1_metrics:
        if 'mape_percent' in f1_metrics and not np.isnan(f1_metrics['mape_percent']):
            interpretations['mape_interpretation'] = (
                f"Mean Absolute Percentage Error: {f1_metrics['mape_percent']:.2f}%. "
                f"This means predictions are on average {f1_metrics['mape_percent']:.2f}% off from actual lap times."
            )

        if 'race_error_50laps_seconds' in f1_metrics:
            interpretations['race_error_interpretation'] = (
                f"Total error over 50-lap race: ±{f1_metrics['race_error_50laps_seconds']:.1f}s. "
                f"This is the cumulative error that would affect race strategy decisions."
            )

        if 'overestimate_pct' in f1_metrics and 'underestimate_pct' in f1_metrics:
            interpretations['directional_interpretation'] = (
                f"Model overestimates (predicts slower) {f1_metrics['overestimate_pct']:.1f}% of the time, "
                f"underestimates (predicts faster) {f1_metrics['underestimate_pct']:.1f}% of the time. "
                f"Systematic bias: {'overestimation' if f1_metrics['overestimate_pct'] > f1_metrics['underestimate_pct'] else 'underestimation'}."
            )

        if 'large_error_rate_pct' in f1_metrics:
            interpretations['large_error_interpretation'] = (
                f"Large errors (>1s) occur {f1_metrics['large_error_rate_pct']:.1f}% of the time. "
                f"These are critical as they can lead to incorrect strategy decisions."
            )

    # Practical implication summary
    if mae < 0.5 and rmse < 1.0 and r2 > 0.7:
        practical = "Model is suitable for F1 strategy decisions with appropriate confidence intervals."
    elif mae < 1.0 and rmse < 2.0 and r2 > 0.5:
        practical = "Model may be useful for rough strategy estimates but requires careful interpretation."
    else:
        practical = "Model accuracy is insufficient for reliable F1 strategy decisions. Consider improving model or data quality."

    interpretations['practical_implication'] = practical

    return interpretations

print("KPI tracking and F1 domain-specific metrics functions loaded")

KPI tracking and F1 domain-specific metrics functions loaded


## Step 4: Check Resources and Configuration

### Quick Testing Mode

For quick error checking, set `TESTING_MODE = True` in the code cell below. This will use:
- **10 iterations** for quantum models (`quantum_maxiter: 10`)
- **15 iterations** for classical HGB model (`hgb_max_iter: 15`)
- **2 CV folds** instead of 5-10
- **Reduced data**: 1 year, 2 events, 200 samples

Set `TESTING_MODE = False` for full training with production settings.

In [8]:
resources = check_resources()

print("\n=== Testing Mode ===")
print(f"TESTING_MODE = {TESTING_MODE}")
if TESTING_MODE:
    print("Quick testing mode: 10-20 iterations for fast error checking")
    print("   Change TESTING_MODE = False above for full training")
else:
    print("Full training mode enabled")

config = get_config()

print("\n=== Configuration ===")
for key, value in config.items():
    if key == 'years':
        print(f"{key}: {value} ({len(value)} years: {min(value)}-{max(value)})")
    else:
        print(f"{key}: {value}")


=== System Resources ===
Platform: Linux
CPU Cores: 12
Total RAM: 167.05 GB
Available RAM: 164.51 GB
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 79.25 GB

=== Testing Mode ===
TESTING_MODE = True
Quick testing mode: 10-20 iterations for fast error checking
   Change TESTING_MODE = False above for full training
TESTING MODE ENABLED - Using reduced iterations (10-20) for quick testing
   Set TESTING_MODE = False for full training
  Year range: 2022-2022 (1 years)

=== Configuration ===
years: [2022] (1 years: 2022-2022)
max_events_per_year: 2
session_name: R
hgb_max_iter: 15
n_splits: 2
quantum_n_components: 4
max_samples: 200
quantum_maxiter: 10
feature_map_reps: 1
ansatz_reps: 1
use_gpu: False


## Step 5: Basic Unit Tests

In [9]:
def test_data_pipeline():
    """Test data pipeline functions."""
    print("Testing data pipeline functions...")

    test_series = pd.Series([pd.Timedelta(seconds=90), pd.Timedelta(seconds=95)])
    result = _timedelta_to_seconds(test_series)
    assert result.dtype == float, "Timedelta conversion failed"
    assert result.iloc[0] == 90.0, "Timedelta conversion incorrect"
    print("  _timedelta_to_seconds")

    test_df = pd.DataFrame({
        'Sector1Time_sec': [30, 31],
        'Sector2Time_sec': [30, 31],
        'Sector3Time_sec': [30, 31],
        'LapTime_sec': [90, 93]
    })
    result_df = _normalize_time_features(test_df)
    assert 'Sector1Ratio' in result_df.columns, "Ratio features not created"
    print("  _normalize_time_features")

    print("Data pipeline tests passed\n")

def test_config_validation():
    """Test configuration validation."""
    print("Testing configuration validation...")

    try:
        invalid_config = {'years': [], 'n_splits': 1}
        validate_config(invalid_config)
        assert False, "Should have raised ValueError"
    except ValueError:
        print("  Invalid config correctly rejected")

    try:
        valid_config = {'years': [2022], 'n_splits': 5, 'quantum_n_components': 4,
                       'max_samples': 100, 'hgb_max_iter': 100, 'quantum_maxiter': 50}
        result = validate_config(valid_config)
        assert result == valid_config, "Valid config should pass unchanged"
        print("  Valid config accepted")
    except Exception as e:
        assert False, f"Valid config rejected: {e}"

    print("Configuration validation tests passed\n")

def test_model_functions():
    """Test model evaluation functions exist and are callable."""
    print("Testing model functions...")

    assert callable(evaluate_classical_models), "evaluate_classical_models not callable"
    assert callable(evaluate_quantum_models), "evaluate_quantum_models not callable"
    assert callable(build_qiskit_regressor), "build_qiskit_regressor not callable"
    assert callable(compare_all_models), "compare_all_models not callable"
    print("  All model functions are callable")

    print("Model function tests passed\n")

print("=" * 80)
print("RUNNING UNIT TESTS")
print("=" * 80)
print()

try:
    test_data_pipeline()
    test_config_validation()
    test_model_functions()
    print("=" * 80)
    print("ALL TESTS PASSED")
    print("=" * 80)
except Exception as e:
    print(f"TEST FAILED: {e}")
    import traceback
    traceback.print_exc()

RUNNING UNIT TESTS

Testing data pipeline functions...
  _timedelta_to_seconds
  _normalize_time_features
Data pipeline tests passed

Testing configuration validation...
  Invalid config correctly rejected
  Valid config accepted
Configuration validation tests passed

Testing model functions...
  All model functions are callable
Model function tests passed

ALL TESTS PASSED


## Step 6: Generate Association Matrix

In [10]:
def load_raw_data_for_profiling(years: List[int], session_name: str = "R",
                                 cache_dir: str = "cache", max_events_per_year: Optional[int] = None,
                                 max_samples: int = 5000):
    """
    Load raw F1 session data before preprocessing for data profiling.
    Returns a combined DataFrame with all raw features.
    """
    # Import fastf1 if not already imported
    try:
        import fastf1
    except ImportError:
        raise ImportError("fastf1 is not installed. Please run: pip install fastf1")

    from pathlib import Path

    Path(cache_dir).mkdir(parents=True, exist_ok=True)
    fastf1.Cache.enable_cache(cache_dir)
    all_laps = []

    try:
        from fastf1.req import RateLimitExceededError
    except ImportError:
        RateLimitExceededError = None

    try:
        fastf1_version = tuple(map(int, fastf1.__version__.split('.')))
    except (AttributeError, ValueError):
        fastf1_version = (3, 0, 0)

    for year in years:
        try:
            schedule = fastf1.get_event_schedule(year)
            time.sleep(0.5)
        except (ValueError, KeyError, AttributeError) as e:
            print(f"  Warning: Could not get schedule for {year}: {e}")
            continue
        except Exception as e:
            error_str = str(e)
            is_rate_limit = (RateLimitExceededError is not None and isinstance(e, RateLimitExceededError)) or \
                           "RateLimit" in error_str or "500 calls" in error_str or "rate limit" in error_str.lower()

            if is_rate_limit:
                print(f"Rate limit exceeded for {year}. Waiting 60 seconds...")
                time.sleep(60)
                try:
                    schedule = fastf1.get_event_schedule(year)
                    time.sleep(0.5)
                except Exception as e2:
                    print(f"  Error: Still cannot get schedule for {year} after wait: {e2}")
                    continue
            else:
                print(f"  Error getting schedule for {year}: {e}")
                continue

        if max_events_per_year is not None:
            schedule = schedule.iloc[:max_events_per_year]
        for _, ev in schedule.iterrows():
            gp_name = ev['EventName']
            try:
                sess = fastf1.get_session(year, gp_name, session_name)
                print(f"Loading {year} {gp_name} {session_name} for profiling...")

                try:
                    if fastf1_version >= (3, 0, 0):
                        try:
                            sess.load(laps=True, telemetry=False, weather=False, messages=False)
                        except TypeError:
                            sess.load(laps=True, telemetry=False, weather=False)
                    elif fastf1_version >= (2, 3, 0):
                        sess.load(laps=True, telemetry=False, weather=False)
                    else:
                        sess.load(with_telemetry=False, weather=False)
                except Exception as load_error:
                    error_str = str(load_error)
                    is_rate_limit = (RateLimitExceededError is not None and isinstance(load_error, RateLimitExceededError)) or \
                                   "RateLimit" in error_str or "500 calls" in error_str or "rate limit" in error_str.lower()

                    if is_rate_limit:
                        print(f"Rate limit during session load. Waiting 60 seconds...")
                        time.sleep(60)
                        if fastf1_version >= (3, 0, 0):
                            try:
                                sess.load(laps=True, telemetry=False, weather=False, messages=False)
                            except TypeError:
                                sess.load(laps=True, telemetry=False, weather=False)
                        elif fastf1_version >= (2, 3, 0):
                            sess.load(laps=True, telemetry=False, weather=False)
                        else:
                            sess.load(with_telemetry=False, weather=False)
                    else:
                        raise

                time.sleep(0.3)

                laps = sess.laps.copy()

                laps['Year'] = year
                laps['EventName'] = gp_name
                laps['Session'] = session_name

                all_laps.append(laps)
            except (ValueError, KeyError, AttributeError) as e:
                print(f"  Data error loading {year} {gp_name} {session_name}: {e}")
                continue
            except (ConnectionError, TimeoutError) as e:
                print(f"  Network error loading {year} {gp_name} {session_name}: {e}")
                continue
            except Exception as e:
                print(f"  Error loading {year} {gp_name} {session_name}: {e}")
                continue

    if not all_laps:
        raise RuntimeError("No data collected. Check years/session_name/filters.")

    df_raw = pd.concat(all_laps, axis=0, ignore_index=True)

    if len(df_raw) > max_samples:
        print(f"Sampling {max_samples} rows from {len(df_raw)} total rows for profiling...")
        df_raw = df_raw.sample(n=max_samples, random_state=42).reset_index(drop=True)

    return df_raw

def generate_simple_correlation_matrix(df: pd.DataFrame, output_file: str = "correlation_matrix.html"):
    """
    Generate a simple correlation matrix using pandas and matplotlib as fallback.
    """

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) < 2:
        print("Warning: Not enough numeric columns for correlation matrix")
        return None

    corr_df = df[numeric_cols].corr()

    plt.figure(figsize=(max(12, len(numeric_cols) * 0.5), max(10, len(numeric_cols) * 0.5)))
    sns.heatmap(corr_df, annot=False, cmap='coolwarm', center=0, square=True,
                fmt='.2f', cbar_kws={"shrink": 0.8}, linewidths=0.5)
    plt.title('Feature Correlation Matrix', fontsize=14, pad=20)
    plt.tight_layout()

    png_file = output_file.replace('.html', '.png')
    plt.savefig(png_file, dpi=150, bbox_inches='tight')
    print(f"Correlation matrix saved to {png_file}")

    corr_df.to_csv(output_file.replace('.html', '.csv'))
    print(f"Correlation data saved to {output_file.replace('.html', '.csv')}")

    plt.close()
    return corr_df

def generate_association_matrix(df: pd.DataFrame, output_file: str = "association_matrix.html"):
    """
    Generate association matrix using ydata-profiling with fallback to simple correlation.
    This shows correlations between numerical features and associations between categorical features.
    Also saves the correlation matrix as a PNG image.
    """
    df_clean = df.copy()

    for col in df_clean.columns:
        if df_clean[col].dtype == 'object':
            try:
                df_clean[col] = df_clean[col].astype(str)
            except:
                df_clean = df_clean.drop(columns=[col])
        elif pd.api.types.is_timedelta64_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].dt.total_seconds()
        elif pd.api.types.is_datetime64_any_dtype(df_clean[col]):
            df_clean = df_clean.drop(columns=[col])

    df_clean = df_clean.select_dtypes(include=[np.number, 'object', 'category'])

    if len(df_clean.columns) == 0:
        print("Warning: No valid columns after cleaning. Using simple correlation matrix...")
        return generate_simple_correlation_matrix(df, output_file)

    # Always generate image version from numeric columns
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) >= 2:
        try:
            corr_matrix = df_clean[numeric_cols].corr()
            png_file = output_file.replace('.html', '.png')

            plt.figure(figsize=(max(14, len(numeric_cols) * 0.6), max(12, len(numeric_cols) * 0.6)))
            sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, square=True,
                       fmt='.2f', cbar_kws={"shrink": 0.8}, linewidths=0.5, vmin=-1, vmax=1)
            plt.title('Feature Correlation Matrix (Association Matrix)', fontsize=16, pad=20, fontweight='bold')
            plt.tight_layout()
            plt.savefig(png_file, dpi=200, bbox_inches='tight')
            plt.close()
            print(f"Association matrix image saved to {png_file}")
        except Exception as e:
            print(f"Note: Could not save association matrix image: {e}")

    try:
        from ydata_profiling import ProfileReport

        print("Generating association matrix with ydata-profiling...")
        print("This may take a few minutes depending on dataset size...")

        try:
            profile = ProfileReport(
                df_clean,
                title="F1 Lap Time Dataset - Association Matrix",
                minimal=True,
                correlations={
                    "pearson": {"calculate": True},
                    "spearman": {"calculate": False},
                    "kendall": {"calculate": False},
                    "phi_k": {"calculate": False},
                    "cramers": {"calculate": True},
                },
                interactions={
                    "continuous": False,
                },
                missing_diagrams={
                    "bar": False,
                    "matrix": False,
                    "heatmap": False,
                },
                duplicates={"head": 0},
                samples={"head": 0, "tail": 0},
            )

            profile.to_file(output_file)
            print(f"Association matrix HTML saved to {output_file}")

            try:
                html_content = profile.to_html()
                with open("association_matrix_standalone.html", "w", encoding="utf-8") as f:
                    f.write(html_content)
                print("Standalone association matrix saved to association_matrix_standalone.html")
            except Exception as e:
                print(f"Note: Could not save standalone matrix: {e}")

            return profile
        except (AttributeError, TypeError, KeyError) as e:
            print(f"Warning: ydata-profiling encountered an error: {e}")
            print("Falling back to simple correlation matrix...")
            return generate_simple_correlation_matrix(df_clean, output_file)
        except Exception as e:
            print(f"Warning: Unexpected error with ydata-profiling: {e}")
            print("Falling back to simple correlation matrix...")
            return generate_simple_correlation_matrix(df_clean, output_file)

    except ImportError:
        print("Warning: ydata-profiling not installed.")
        print("Using simple correlation matrix instead...")
        return generate_simple_correlation_matrix(df_clean, output_file)

print("Skipping association matrix generation (A1 minimal output workflow).")

Skipping association matrix generation (A1 minimal output workflow).


## Step 7: Run Full Model Comparison

In [11]:
import gc
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

gc.collect()

# =========================
# A1 minimal output workflow
# =========================

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

MODEL_ORDER = ["ridge", "ridge_pca", "hgb", "qiskit_qnn"]
MODEL_LABEL = {
    "ridge": "Ridge Regression",
    "ridge_pca": "Ridge PCA",
    "hgb": "Histogram Gradient Boosting",
    "qiskit_qnn": "Quantum Neural Network"
}


def _truncate_label(s: str, n: int = 26) -> str:
    s = str(s)
    return s if len(s) <= n else (s[: n - 3] + "...")


def _savefig(fig, filename: str):
    # Centralized save style for publication-like outputs
    fig.patch.set_facecolor("white")
    fig.savefig(filename, dpi=200, bbox_inches="tight")


# Generating A1 final outputs
# Years/Session/CV folds (see config above)

try:
    comparison_df, results = compare_all_models(
        years=config['years'],
        session_name=config['session_name'],
        max_events_per_year=config['max_events_per_year'],
        quantum_n_components=config['quantum_n_components'],
        quantum_max_samples=config['max_samples'],
        n_splits=config['n_splits'],
        quantum_maxiter=config['quantum_maxiter'],
        hgb_max_iter=config['hgb_max_iter'],
        use_gpu=config['use_gpu'],
        quantum_feature_map_reps=config['feature_map_reps'],
        quantum_ansatz_reps=config['ansatz_reps'],
        save_plots=False,  # Disable legacy/duplicate outputs
        min_laps_per_driver=5
    )

    classical_results = results["classical"]["results"]
    quantum_results = results["quantum"]["results"]
    classical_errors = results["classical"]["errors"]
    quantum_errors = results["quantum"]["errors"]

    all_results = pd.concat([classical_results, quantum_results], ignore_index=True)
    all_errors = pd.concat([classical_errors, quantum_errors], ignore_index=True)

except Exception as e:
    print(f"\nERROR during model comparison: {e}")
    import traceback
    traceback.print_exc()
    raise


# =========================
# Load dataset for feature/tyre plots + dataset feature summary
# (sample limit matches the evaluation max_samples for fair alignment)
# =========================

X_classical, _, y, meta, _, _ = load_sessions_and_build_dataset(
    years=config['years'],
    session_name=config['session_name'],
    cache_dir="cache",
    max_events_per_year=config['max_events_per_year'],
    quantum_n_components=config['quantum_n_components'],
    min_laps_per_driver=5
)

max_samples = config.get('max_samples', None)
if max_samples and len(X_classical) > max_samples:
    rng = np.random.RandomState(42)
    idx = rng.choice(len(X_classical), size=max_samples, replace=False)
    idx = np.sort(idx)
    X_classical = X_classical.iloc[idx].reset_index(drop=True)
    y = y.iloc[idx].reset_index(drop=True)
    meta = meta.iloc[idx].reset_index(drop=True)
else:
    # Ensure clean indexing for consistent plotting
    X_classical = X_classical.reset_index(drop=True)
    y = y.reset_index(drop=True)
    meta = meta.reset_index(drop=True)

meta_plot = meta.copy()
meta_plot["LapTime_sec"] = y.values


# =========================
# MAIN REPORT: Figures
# =========================

# ---- MAIN REPORT: figure_1_feature_correlation_matrix.png ----
# MAIN REPORT: Association / correlation matrix (Top 20 Features)
# figure_1_feature_correlation_matrix.png

# Build a Top-N feature correlation matrix for readability (rotated x-axis labels)
numeric_cols = [c for c in X_classical.columns if pd.api.types.is_numeric_dtype(X_classical[c])]
if len(numeric_cols) < 2:
    print("Warning: Not enough numeric columns for association/correlation matrix; skipping.")
else:
    # Rank features by absolute correlation with target
    target_corrs = []
    for c in numeric_cols:
        try:
            corr = abs(X_classical[c].corr(y))
            if np.isfinite(corr):
                target_corrs.append((c, corr))
        except Exception:
            pass

    target_corrs.sort(key=lambda t: t[1], reverse=True)
    top_features = [c for c, _ in target_corrs[:min(20, len(target_corrs))]]

    if len(top_features) < 2:
        print("Warning: Top feature selection resulted in <2 features; skipping association matrix.")
    else:
        corr_df = X_classical[top_features].corr()

        # Mask upper triangle to reduce clutter
        mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)

        fig, ax = plt.subplots(figsize=(16, 12))
        sns.heatmap(
            corr_df,
            mask=mask,
            annot=False,
            cmap="RdBu_r",
            center=0,
            square=True,
            linewidths=0.5,
            vmin=-1,
            vmax=1,
            cbar_kws={"shrink": 0.85},
            ax=ax,
        )

        labels = [_truncate_label(c, n=24) for c in top_features]
        ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=9)
        ax.set_yticklabels(labels, rotation=0, fontsize=9)
        ax.set_title(f"Feature Correlation Matrix (Top {len(top_features)} Features)")

        fig.tight_layout()
        _savefig(fig, "figure_1_feature_correlation_matrix.png")
        plt.close(fig)

# ---- MAIN REPORT: figure_2_feature_importance.png ----
# figure_2_feature_importance.png

corr_rows = []
for col in X_classical.columns:
    if pd.api.types.is_numeric_dtype(X_classical[col]):
        try:
            corr = np.abs(X_classical[col].corr(y))
            if np.isfinite(corr):
                corr_rows.append({"Feature": col, "Abs_Correlation": corr})
        except Exception:
            pass

corr_df = pd.DataFrame(corr_rows)
if corr_df.empty:
    print("Warning: No usable feature correlations found; skipping figure_1.")
else:
    corr_df = corr_df.sort_values("Abs_Correlation", ascending=False).head(15)
    feature_labels = [_truncate_label(f) for f in corr_df["Feature"].values]

    fig, ax = plt.subplots(figsize=(10.5, 6))
    ax.barh(range(len(corr_df)), corr_df["Abs_Correlation"].values, color="#2E86AB", alpha=0.85)
    ax.set_yticks(range(len(corr_df)))
    ax.set_yticklabels(feature_labels)
    ax.invert_yaxis()
    ax.set_xlabel("Absolute Correlation with Lap Time")
    ax.set_ylabel("")
    ax.set_title("Top Features by Correlation with Lap Time")
    _savefig(fig, "figure_2_feature_importance.png")
    plt.close(fig)


# ---- MAIN REPORT: figure_3_tyre_life_vs_lap_time.png ----
# figure_3_tyre_life_vs_lap_time.png

if "TyreLife" not in meta_plot.columns:
    print("Warning: 'TyreLife' column missing in meta; skipping figure_2.")
else:
    tyre_df = meta_plot[["TyreLife", "LapTime_sec"]].dropna()
    tyre_df = tyre_df[(tyre_df["LapTime_sec"] > 0) & np.isfinite(tyre_df["LapTime_sec"]) & np.isfinite(tyre_df["TyreLife"]) ]

    fig, ax = plt.subplots(figsize=(10.5, 6))
    ax.scatter(tyre_df["TyreLife"], tyre_df["LapTime_sec"], s=10, alpha=0.35, color="#A23B72")

    if len(tyre_df) > 3:
        try:
            z = np.polyfit(tyre_df["TyreLife"].values, tyre_df["LapTime_sec"].values, 1)
            p = np.poly1d(z)
            x_line = np.linspace(tyre_df["TyreLife"].min(), tyre_df["TyreLife"].max(), 200)
            ax.plot(x_line, p(x_line), color="black", linewidth=2)
        except Exception:
            pass

    ax.set_xlabel("Tyre Life")
    ax.set_ylabel("Lap Time (s)")
    ax.set_title("Tyre Life vs Lap Time (Degradation Effect)")
    _savefig(fig, "figure_3_tyre_life_vs_lap_time.png")
    plt.close(fig)


# ---- MAIN REPORT: metric bars + trade-off + error distribution ----
# figure_3_to_figure_8

perf_means = all_results.groupby("model").agg(
    MAE=("MAE", "mean"),
    RMSE=("RMSE", "mean"),
    R2=("R2", "mean"),
    TrainTime=("train_time_s", "mean"),
    InferTime=("infer_time_s", "mean"),
).reset_index()

# Ensure consistent order
perf_means = perf_means.set_index("model").reindex(MODEL_ORDER).reset_index()
perf_means["Model"] = perf_means["model"].map(MODEL_LABEL)


def _bar_metric(col, filename, ylabel, title):
    fig, ax = plt.subplots(figsize=(10.5, 6))
    plot_df = perf_means[["Model", col]].copy()
    sns.barplot(data=plot_df, x="Model", y=col, ax=ax, order=plot_df["Model"].tolist(), color="#2E86AB", alpha=0.85)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=25)
    _savefig(fig, filename)
    plt.close(fig)


_bar_metric("MAE", "figure_4_mae_comparison.png", "MAE (s)", "MAE Comparison by Model")
_bar_metric("RMSE", "figure_5_rmse_comparison.png", "RMSE (s)", "RMSE Comparison by Model")
_bar_metric("R2", "figure_6_r2_comparison.png", "R²", "R² Comparison by Model")
_bar_metric("TrainTime", "figure_7_training_time_comparison.png", "Training Time (s)", "Training Time Comparison by Model")


# ---- MAIN REPORT: figure_8_accuracy_vs_training_time.png ----
fig, ax = plt.subplots(figsize=(10.5, 6))
ax.scatter(perf_means["TrainTime"], perf_means["MAE"], s=140, alpha=0.9, color="#2E86AB")
for _, row in perf_means.iterrows():
    ax.annotate(row["Model"], (row["TrainTime"], row["MAE"]), textcoords="offset points", xytext=(6, 4), fontsize=10)
ax.set_xlabel("Training Time (s)")
ax.set_ylabel("MAE (s)")
ax.set_title("Accuracy vs Training Time Trade-off")
ax.set_xscale("log")
ax.grid(True, axis='both', alpha=0.25)
_savefig(fig, "figure_8_accuracy_vs_training_time.png")
plt.close(fig)


# ---- MAIN REPORT: figure_9_error_distribution_comparison.png ----
# figure_9_error_distribution_comparison.png
if "abs_error_s" not in all_errors.columns:
    print("Warning: 'abs_error_s' missing in all_errors; skipping figure_8.")
else:
    err_df = all_errors.copy()
    err_df["Model"] = err_df["model"].map(MODEL_LABEL)

    fig, ax = plt.subplots(figsize=(10.5, 6))
    sns.boxplot(
        data=err_df,
        x="Model",
        y="abs_error_s",
        order=[MODEL_LABEL[m] for m in MODEL_ORDER],
        ax=ax,
        color="#A23B72",
        showfliers=False
    )
    ax.set_xlabel("")
    ax.set_ylabel("Absolute Error (s)")
    ax.set_title("Prediction Error Distribution Across Models")
    ax.tick_params(axis='x', rotation=25)
    _savefig(fig, "figure_9_error_distribution_comparison.png")
    plt.close(fig)


# =========================
# MAIN REPORT: Tables
# =========================

# table_1_to_table_3

# ---- table_1_model_configurations.csv ----
rows = [
    {"Model": MODEL_LABEL["ridge"], "Type": "Classical ML", "Feature Space / Input Representation": "Full telemetry feature set"},
    {"Model": MODEL_LABEL["ridge_pca"], "Type": "Classical ML", "Feature Space / Input Representation": f"PCA-reduced telemetry features (n_components={config['quantum_n_components']})"},
    {"Model": MODEL_LABEL["hgb"], "Type": "Classical ML", "Feature Space / Input Representation": "Full telemetry feature set"},
    {"Model": MODEL_LABEL["qiskit_qnn"], "Type": "Quantum ML", "Feature Space / Input Representation": f"PCA-reduced features encoded into a ZZFeatureMap + RealAmplitudes circuit (n_components={config['quantum_n_components']})"},
]

table1 = pd.DataFrame(rows)
table1.to_csv("table_1_model_configurations.csv", index=False)


# ---- table_2_predictive_performance_summary.csv ----
perf_table = perf_means[["Model", "MAE", "RMSE", "R2"]].copy()
perf_table[["MAE", "RMSE", "R2"]] = perf_table[["MAE", "RMSE", "R2"]].round(4)
perf_table.to_csv("table_2_predictive_performance_summary.csv", index=False)


# ---- table_3_computational_performance_summary.csv ----
comp_table = perf_means[["Model", "TrainTime", "InferTime"]].copy()
comp_table["Computational Cost"] = comp_table["TrainTime"] + comp_table["InferTime"]
comp_table = comp_table.rename(columns={"TrainTime": "Training Time", "InferTime": "Inference Time"})
comp_table[["Training Time", "Inference Time", "Computational Cost"]] = comp_table[["Training Time", "Inference Time", "Computational Cost"]].round(4)
comp_table.to_csv("table_3_computational_performance_summary.csv", index=False)


# =========================
# APPENDIX outputs (combined where required)
# =========================

# ---- appendix_a1_a2_data_exploration.png ----
# appendix_a1_a2_data_exploration.png
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y.values, bins=50, edgecolor="black", alpha=0.75, color="steelblue")
axes[0].set_title("Lap Time Distribution")
axes[0].set_xlabel("Lap Time (s)")
axes[0].set_ylabel("Count")

if "Compound" in meta_plot.columns:
    cdf = meta_plot[["Compound", "LapTime_sec"]].dropna()
    top_compounds = cdf["Compound"].value_counts().head(6).index
    cdf = cdf[cdf["Compound"].isin(top_compounds)]

    sns.boxplot(data=cdf, x="Compound", y="LapTime_sec", ax=axes[1], palette="Set2")
    axes[1].set_title("Lap Time by Tyre Compound")
    axes[1].tick_params(axis='x', rotation=25)
    axes[1].set_xlabel("Tyre Compound")
    axes[1].set_ylabel("Lap Time (s)")
else:
    axes[1].text(0.5, 0.5, "No tyre compound data available", ha="center", va="center")
    axes[1].set_axis_off()

_savefig(fig, "appendix_a1_a2_data_exploration.png")
plt.close(fig)


# ---- appendix_b1_b2_cross_validation_metrics.png ----
# appendix_b1_b2_cross_validation_metrics.png

cv = all_results.copy()
cv["Model"] = cv["model"].map(MODEL_LABEL)
cv = cv[cv["model"].isin(MODEL_ORDER)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(data=cv, x="fold", y="MAE", hue="Model", hue_order=[MODEL_LABEL[m] for m in MODEL_ORDER], marker="o", ax=axes[0])
axes[0].set_title("Cross-validation MAE Across Folds")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("MAE (s)")

sns.lineplot(data=cv, x="fold", y="R2", hue="Model", hue_order=[MODEL_LABEL[m] for m in MODEL_ORDER], marker="o", ax=axes[1])
axes[1].set_title("Cross-validation R² Across Folds")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("R²")

axes[0].legend(title="Model", fontsize=9)
axes[1].legend(title="Model", fontsize=9)

_savefig(fig, "appendix_b1_b2_cross_validation_metrics.png")
plt.close(fig)


# ---- appendix_b3_b5_contextual_error_analysis.png ----
# appendix_b3_b5_contextual_error_analysis.png

err_ctx = all_errors.copy()
err_ctx["Model"] = err_ctx["model"].map(MODEL_LABEL)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) Error by driver
if "Driver" in err_ctx.columns:
    top_drivers = err_ctx["Driver"].value_counts().head(12).index
    driver_df = err_ctx[err_ctx["Driver"].isin(top_drivers)].groupby(["Model", "Driver"])["abs_error_s"].mean().reset_index()
    driver_df = driver_df.sort_values("abs_error_s", ascending=False)

    sns.barplot(data=driver_df, x="Driver", y="abs_error_s", hue="Model", order=top_drivers.tolist(), ax=axes[0], palette="deep")
    axes[0].set_title("Error by Driver")
    axes[0].set_xlabel("Driver")
    axes[0].set_ylabel("Mean Absolute Error (s)")
    axes[0].tick_params(axis='x', rotation=45)
else:
    axes[0].text(0.5, 0.5, "No driver column", ha="center", va="center")
    axes[0].set_axis_off()

# (2) Error by tyre compound
if "Compound" in err_ctx.columns:
    top_compounds = err_ctx["Compound"].value_counts().head(6).index
    compound_df = err_ctx[err_ctx["Compound"].isin(top_compounds)].groupby(["Model", "Compound"])["abs_error_s"].mean().reset_index()

    sns.barplot(data=compound_df, x="Compound", y="abs_error_s", hue="Model", order=top_compounds.tolist(), ax=axes[1], palette="deep")
    axes[1].set_title("Error by Tyre Compound")
    axes[1].set_xlabel("Tyre Compound")
    axes[1].set_ylabel("Mean Absolute Error (s)")
    axes[1].tick_params(axis='x', rotation=25)
else:
    axes[1].text(0.5, 0.5, "No compound column", ha="center", va="center")
    axes[1].set_axis_off()

# (3) Error by race position
if "Position" in err_ctx.columns:
    position_df = err_ctx[["Model", "Position", "abs_error_s"]].dropna()
    position_df["PositionBin"] = pd.cut(
        position_df["Position"],
        bins=[0, 5, 10, 15, 20, 25],
        labels=["1-5", "6-10", "11-15", "16-20", "21+"],
        include_lowest=True
    )
    pos_df = position_df.groupby(["Model", "PositionBin"])["abs_error_s"].mean().reset_index()

    sns.barplot(data=pos_df, x="PositionBin", y="abs_error_s", hue="Model", ax=axes[2], palette="deep")
    axes[2].set_title("Error by Race Position")
    axes[2].set_xlabel("Race Position")
    axes[2].set_ylabel("Mean Absolute Error (s)")
else:
    axes[2].text(0.5, 0.5, "No Position column", ha="center", va="center")
    axes[2].set_axis_off()

if axes[0].get_legend() is not None:
    axes[0].legend(title="Model", fontsize=9)
if axes[1].get_legend() is not None:
    axes[1].get_legend().remove()
if axes[2].get_legend() is not None:
    axes[2].get_legend().remove()

_savefig(fig, "appendix_b3_b5_contextual_error_analysis.png")
plt.close(fig)


# ---- appendix_b6_training_time_distribution.png ----
# appendix_b6_training_time_distribution.png
fig, ax = plt.subplots(figsize=(10.5, 6))
for m in MODEL_ORDER:
    dd = all_results[all_results["model"] == m]["train_time_s"].dropna().values
    if len(dd) == 0:
        continue
    sns.histplot(dd, bins=25, element="step", stat="density", alpha=0.35, ax=ax, label=MODEL_LABEL[m])
ax.set_xscale("log")
ax.set_title("Training Time Distribution")
ax.set_xlabel("Training Time (s) [log scale]")
ax.set_ylabel("Density")
ax.legend(fontsize=9, title="Model")
_savefig(fig, "appendix_b6_training_time_distribution.png")
plt.close(fig)


# ---- appendix_b7_inference_time_distribution.png ----
# appendix_b7_inference_time_distribution.png
fig, ax = plt.subplots(figsize=(10.5, 6))
for m in MODEL_ORDER:
    dd = all_results[all_results["model"] == m]["infer_time_s"].dropna().values
    if len(dd) == 0:
        continue
    sns.histplot(dd, bins=25, element="step", stat="density", alpha=0.35, ax=ax, label=MODEL_LABEL[m])
ax.set_xscale("log")
ax.set_title("Inference Time Distribution")
ax.set_xlabel("Inference Time (s) [log scale]")
ax.set_ylabel("Density")
ax.legend(fontsize=9, title="Model")
_savefig(fig, "appendix_b7_inference_time_distribution.png")
plt.close(fig)


# =========================
# Optional appendix tables (saved once)
# =========================
# optional appendix tables

b1 = all_results.copy()
b1["Model"] = b1["model"].map(MODEL_LABEL)
keep_cols = ["Model", "fold", "MAE", "RMSE", "R2", "train_time_s", "infer_time_s"]
b1 = b1[keep_cols].rename(columns={
    "fold": "Fold",
    "train_time_s": "Training Time (s)",
    "infer_time_s": "Inference Time (s)"
})
for c in ["MAE", "RMSE", "R2", "Training Time (s)", "Inference Time (s)"]:
    b1[c] = b1[c].round(6)
b1.to_csv("appendix_table_b1_detailed_model_results.csv", index=False)

b2 = classical_results.copy()
b2["Model"] = b2["model"].map(MODEL_LABEL)
keep_cols = ["Model", "fold", "MAE", "RMSE", "R2", "train_time_s", "infer_time_s"]
b2 = b2[keep_cols].rename(columns={
    "fold": "Fold",
    "train_time_s": "Training Time (s)",
    "infer_time_s": "Inference Time (s)"
})
for c in ["MAE", "RMSE", "R2", "Training Time (s)", "Inference Time (s)"]:
    b2[c] = b2[c].round(6)
b2.to_csv("appendix_table_b2_classical_fold_metrics.csv", index=False)

b3 = quantum_results.copy()
b3["Model"] = b3["model"].map(MODEL_LABEL)
keep_cols = ["Model", "fold", "MAE", "RMSE", "R2", "train_time_s", "infer_time_s"]
b3 = b3[keep_cols].rename(columns={
    "fold": "Fold",
    "train_time_s": "Training Time (s)",
    "infer_time_s": "Inference Time (s)"
})
for c in ["MAE", "RMSE", "R2", "Training Time (s)", "Inference Time (s)"]:
    b3[c] = b3[c].round(6)
b3.to_csv("appendix_table_b3_quantum_fold_metrics.csv", index=False)

feat_rows = []
for col in X_classical.columns:
    if pd.api.types.is_numeric_dtype(X_classical[col]):
        try:
            corr = np.abs(X_classical[col].corr(y))
            if np.isfinite(corr):
                feat_rows.append({"Feature": col, "Abs_Correlation_with_LapTime": corr})
        except Exception:
            pass

feat_tbl = pd.DataFrame(feat_rows).sort_values("Abs_Correlation_with_LapTime", ascending=False)
feat_tbl["Abs_Correlation_with_LapTime"] = feat_tbl["Abs_Correlation_with_LapTime"].round(6)
feat_tbl.to_csv("appendix_table_e1_dataset_feature_summary.csv", index=False)

print("Final outputs generated.")

Step 1: Evaluating Classical Models


Loading years:   0%|          | 0/1 [00:00<?, ?it/s]

  2022 events:   0%|          | 0/2 [00:00<?, ?it/s]

events      WARNING 	Correcting user input 'Pre-Season Track Session' to 'British Grand Prix'
core           INFO 	Loading data for British Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for British Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-


Dataset: 1383 samples, 59 features
Successfully loaded 2 sessions from 1 years
Preprocessed data cached to cache/preprocessed_2022_2022_R_4.pkl
Applied max_samples=200 limit for fair comparison with quantum models
Evaluating classical models: 200 samples, 2 CV folds
  Added PCA-reduced baseline (n_components=4) for fair comparison
Training RIDGE...


...

  ridge_pca folds:   0%|          | 0/2 [00:00<?, ?it/s]


=== Aggregate fold metrics ===
                MAE                RMSE                   R2            \
               mean       std      mean       std       mean       std   
model                                                                    
hgb        8.398661  0.111666  8.507683  0.124291 -38.500117  3.974864   
ridge      3.282588  2.771388  3.306524  2.766286  -6.420615  8.955014   
ridge_pca  4.360290  0.064463  4.458954  0.045803  -9.855827  1.186069   

          train_time_s           infer_time_s            
                  mean       std         mean       std  
model                                                    
hgb           0.056567  0.039187     0.001801  0.000153  
ridge         0.003354  0.000781     0.001626  0.000051  
ridge_pca     0.001975  0.000308     0.000919  0.000033  

Step 2: Evaluating Quantum Models
Loading cached preprocessed data from cache/preprocessed_2022_2022_R_4.pkl...
Cached data loaded successfully
Evaluating quantum model: 200 

  Quantum folds:   0%|          | 0/2 [00:00<?, ?it/s]

  Using standard Estimator
  Using StatevectorEstimator (recommended for EstimatorQNN compatibility)
  Fold 1: Training...


  Using standard Estimator
  Using StatevectorEstimator (recommended for EstimatorQNN compatibility)
  Fold 2: Training...

=== Aggregate fold metrics ===
                 MAE               RMSE                   R2            \
                mean       std     mean       std       mean       std   
model                                                                    
qiskit_qnn  9.049769  0.033665  9.15472  0.062592 -44.779172  5.314181   

           train_time_s           infer_time_s            
                   mean       std         mean       std  
model                                                     
qiskit_qnn     1.778479  0.391756      0.17272  0.043774  

Step 3: Generating Comparison Report

Comparison Summary:
               MAE            RMSE               R2         train_time_s  \
              mean     std    mean     std     mean     std         mean   
model                                                                      
hgb         8.3987  0.111

## Step 8: Generate Visualizations

In [12]:
print("Skipping legacy visualization generation (A1 minimal output workflow). Final outputs are generated in the model-comparison cell.")

Skipping legacy visualization generation (A1 minimal output workflow). Final outputs are generated in the model-comparison cell.


## Step 9: Download Results (Colab Only)

In [13]:
FINAL_FILES = [
    # Main report figures
    'figure_1_feature_correlation_matrix.png',
    'figure_2_feature_importance.png',
    'figure_3_tyre_life_vs_lap_time.png',
    'figure_4_mae_comparison.png',
    'figure_5_rmse_comparison.png',
    'figure_6_r2_comparison.png',
    'figure_7_training_time_comparison.png',
    'figure_8_accuracy_vs_training_time.png',
    'figure_9_error_distribution_comparison.png',

    # Appendix figures (combined outputs)
    'appendix_a1_a2_data_exploration.png',
    'appendix_b1_b2_cross_validation_metrics.png',
    'appendix_b3_b5_contextual_error_analysis.png',
    'appendix_b6_training_time_distribution.png',
    'appendix_b7_inference_time_distribution.png',

    # Main tables
    'table_1_model_configurations.csv',
    'table_2_predictive_performance_summary.csv',
    'table_3_computational_performance_summary.csv',

    # Optional appendix tables
    'appendix_table_b1_detailed_model_results.csv',
    'appendix_table_b2_classical_fold_metrics.csv',
    'appendix_table_b3_quantum_fold_metrics.csv',
    'appendix_table_e1_dataset_feature_summary.csv',
]

if IN_COLAB:
    from google.colab import files
    import os

    missing = [f for f in FINAL_FILES if not os.path.exists(f)]
    if missing:
        print(f"Missing before download ({len(missing)} files):")
        for f in missing:
            print(" -", f)
    else:
        print("All FINAL_FILES present before download.")

    # Bundle all outputs into a single ZIP for reliable bulk download
    import zipfile
    zip_name = "A1_TestMode_FINAL_FILES.zip"
    with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for f in FINAL_FILES:
            if not os.path.exists(f):
                print(f"Note: {f} not found; skipping into zip")
                continue
            zf.write(f)

    print(f"ZIP created: {zip_name}")

    try:
        files.download(zip_name)
        print(f"{zip_name} downloaded")
    except Exception as e:
        print(f"Note: {zip_name} could not be downloaded: {e}")
else:
    print("A1 final outputs saved locally:")
    for f in FINAL_FILES:
        print("  -", f)


All FINAL_FILES present before download.
ZIP created: A1_TestMode_FINAL_FILES.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

A1_TestMode_FINAL_FILES.zip downloaded
